# Config Optimizer — Best Empire Configuration (Genetic Algorithm)

Which building levels, on which planets, build the most Imperial starbases per month? This notebook searches for that configuration with a genetic algorithm and saves what it finds as a JSON file.

**Inputs** (read-only):

- `state/spaceinvasion.db` — your planets, your research levels and the research levels known in the universe, always read through a temporary copy so the live database is never opened;
- `spaceinvasion.core.game_data` — the wiki catalog of costs, build times, production and requirements;
- `notebooks/metadata/*.json` — static descriptions and planner settings that are not part of the catalog.

**Output**: `notebooks/metadata/config_optimizer_result.json`, written by the last cell. It is the only file that the notebook writes, and every run replaces it.

**Layout**: Section 0 installs the packages and imports the modules. Section 1 holds the meta parameters: everything you can tune. Section 2 holds the helpers. Section 3 loads the inputs: your planets, padded with generic planets up to the planet limit, and the research levels. Section 4 is the optimizer: the plan inputs, the production model, the search and the result. Section 5 saves the result.

The helpers of Section 2 and the loading statements of Section 3 are copies from `full_planner.ipynb`, limited to what this notebook needs. A unit test fails when a copy differs from its original, so a shared helper has to be changed in both notebooks.

## 0. Package Installation

Installs the third-party packages this notebook imports, directly or through the `spaceinvasion` package: `numpy`, `pandas` and `beautifulsoup4`. The version ranges are the ones in `requirements.txt`. When the packages are already installed, the cell changes nothing.

In [ ]:
!pip install -q "numpy>=1.26,<3" "pandas>=2.2,<4" "beautifulsoup4>=4.12,<5"

## 0.1 Imports

Every import the notebook uses is in this cell. The loop before the `spaceinvasion` imports puts the repository root on `sys.path`, so the notebook also runs when it is opened from the `notebooks/` folder.

In [ ]:
from collections import deque
from collections.abc import Iterable, Iterator
import contextlib
import copy
import dataclasses
import datetime
import json
import math
from pathlib import Path
import shutil
import sqlite3
import sys
import tempfile
from typing import Any

import numpy as np
import pandas as pd

# Make the repository importable when the notebook is opened from `notebooks/`.
for _cand in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (_cand / "spaceinvasion" / "__init__.py").is_file():
        if str(_cand) not in sys.path:
            sys.path.insert(0, str(_cand))
        break

from spaceinvasion.combat.fleet import DEFENSE_SLUG_BY_KEY, SHIP_SLUG_BY_KEY
from spaceinvasion.core import game_data
from spaceinvasion.core.logging_utils import redact_sensitive
from spaceinvasion.core.state import EnergyBalance, PlanetState, Resources
from spaceinvasion.db.storage import GameDatabase

## 1. Meta Parameters

Everything you can tune is in this cell, and nothing else: it only sets values. Section 4.1 checks them and prints them together with the plan that follows from them.

- `TARGET_SHIP_KEY`, `PRODUCTION_MONTH_DAYS` — the ship to count, and the length of the month.
- `RESEARCH_PERCENT_OF_UNIVERSE_MAX` — the research levels of the plan, as a percentage of the universe maximum of Section 3.2. Use a lower percentage when the universe maximum is far beyond your current game. A planned level never falls below your current level or below a level that the plan requires.
- `MIN_FLEET_PER_PLANET` — ships that every planet keeps. The plan gives every planet the Weapons factory level needed to build them, counts the energy of Solar cells among them, and reports what the missing ships cost.
- `MISC_BUILDING_LEVELS`, `MISC_BUILDING_LEVELS_BY_SLOT` — the level of every building that the algorithm does not decide: on all planets, and exceptions for single planets. A colony keeps a level that it has already built beyond these. For the four warehouses the level is only a minimum.
- `WAREHOUSE_HOURS_OF_PRODUCTION` — the hours of production that the warehouses hold (rules 1 and 3 of Section 4).
- `CENTRAL_PLANET_SLOT`, `CENTRAL_PLANET_MIN_WAREHOUSE_LEVEL` — the roster slot of the central planet, your home planet unless you change it, and the lowest level of its four warehouses (rule 3 of Section 4).
- `OPTIMIZED_BUILDING_MAX_LEVELS` — the buildings whose levels the algorithm decides, each with the highest level it may choose. Together with the miscellaneous buildings they must cover every building once.
- `CATALOGUE_*`, `HILL_CLIMB_STEP_SIZES`, `GA_*` — settings of the search, explained in Section 4.3.

In [ ]:
# --- Objective -------------------------------------------------------------------------------------------------
TARGET_SHIP_KEY: str = "s16"  # the ship to mass-produce: the Imperial starbase
PRODUCTION_MONTH_DAYS: float = 30.0  # length of the month that the production is counted over

# --- Research reference ----------------------------------------------------------------------------------------
# Research levels of the plan, as a percentage of the highest level known in the universe (Section 3.2).
# 100 plans with the universe maximum; lower it to stay closer to your current game. A planned level never falls
# below your current level, nor below the level that the ships and buildings of the plan require.
RESEARCH_PERCENT_OF_UNIVERSE_MAX: float = 100.0

# --- Fixed part of every planet --------------------------------------------------------------------------------
# Ships that every planet keeps (ship key -> count). Every planet must be able to build them itself.
MIN_FLEET_PER_PLANET: dict[str, int] = {
    "s2": 10,  # Large transport
    "s10": 10,  # Spy probe
    "s11": 10,  # Recycler
}
# Level of the miscellaneous buildings, the ones the genetic algorithm does not decide, on every planet.
# For the four warehouses it is only the lowest level: the plan raises each one to what its planet needs.
MISC_BUILDING_LEVELS: dict[str, int] = {
    "b6": 0,  # Thermosolar plant
    "b7": 12,  # Development center (the Microsystem accelerator requires level 12)
    "b8": 0,  # Research lab
    "b10": 5,  # Fleet base
    "b11": 0,  # Pig iron warehouse
    "b12": 0,  # Metal warehouse
    "b13": 0,  # Kryptonite warehouse
    "b14": 0,  # Spice warehouse
    "b15": 0,  # Particle cannon
    "b16": 0,  # Particle shield
    "b18": 0,  # Teleporter
    "b19": 0,  # Galaxy scanner
    "b20": 0,  # Tunnel system
    "b21": 0,  # Quantum Array
}
# Exceptions for single planets (roster slot -> building key -> level); slot 1 is the first planet of the roster.
MISC_BUILDING_LEVELS_BY_SLOT: dict[int, dict[str, int]] = {
    1: {"b8": 12},  # one Research lab for the whole empire
}

# --- Warehouses ------------------------------------------------------------------------------------------------
# Every warehouse gets the lowest level that holds this many hours of its planet's own production, and the cost of
# the most expensive thing that the planet still has to pay for: a building level, a ship or, on the planet with
# the Research lab, a research level.
WAREHOUSE_HOURS_OF_PRODUCTION: float = 24.0
# The central planet collects the production of the empire. Its warehouses also hold that many hours of the
# production of all planets together, and none of the four falls below the minimum level.
CENTRAL_PLANET_SLOT: int = 1  # roster slot of the central planet: the home planet
CENTRAL_PLANET_MIN_WAREHOUSE_LEVEL: int = 15  # lowest level of each of its four warehouses

# --- What the genetic algorithm decides ------------------------------------------------------------------------
# Buildings whose level it chooses on every planet, each with the highest level it may choose. A maximum beyond
# the last level of the catalog (100 for the mines and the Fusion plant, 60 for the Weapons factory, 55 for the
# Microsystem accelerator) is cut to that level.
OPTIMIZED_BUILDING_MAX_LEVELS: dict[str, int] = {
    "b1": 100,  # Pig iron mine
    "b2": 100,  # Blast furnace
    "b3": 100,  # Kryptonite mine
    "b4": 100,  # Spice mine
    "b5": 100,  # Fusion plant
    "b9": 100,  # Weapons factory
    "b17": 100,  # Microsystem accelerator
}

# --- Search settings -------------------------------------------------------------------------------------------
CATALOGUE_GRID_SIZE: int = 100_000  # coarse designs tried per planet before its design catalogue is refined
CATALOGUE_PRIORITY_STEPS: int = 4  # priorities between the limits vary in steps of 1/4
HILL_CLIMB_STEP_SIZES: tuple[int, ...] = (1, 2, 4, 8)  # level changes that the hill-climb tries
GA_POPULATION_SIZE: int = 400  # chromosomes per generation
GA_GENERATIONS: int = 300  # generations to evolve
GA_TOURNAMENT_SIZE: int = 3  # chromosomes compared to choose one parent
GA_CROSSOVER_RATE: float = 0.9  # share of children that mix two parents
GA_MUTATION_RATE: float = 0.1  # chance that a planet of a child switches to a random design
GA_ELITE_COUNT: int = 4  # best chromosomes copied unchanged into the next generation
GA_RANDOM_SEED: int = 20261117  # the same seed gives the same result

## 2. Utils

Helpers of the sections below, one purpose per cell. They are the helpers of `full_planner.ipynb`, limited to what this notebook needs.

### 2.1 Full-Data Printing

`print_section_data` prints every row, column and element of the objects it is given, under a header with the date and time of the printout. Every cell of this notebook that generates data ends by printing that data with it. Floats of 1 or more are never switched to scientific notation, and all text passes through `redact_sensitive`, so a session token can never appear in a printout.

In [ ]:
def format_full_float(value: float) -> str:
    """Format a float for full printouts without scientific notation for large values.

    `|x| >= 1` keeps its shortest exact decimal text, capped at 6 decimals; `|x| < 1` keeps 6 significant digits.
    """
    val = float(value)
    if not math.isfinite(val):
        return str(val)
    if val == 0.0:
        return "0"
    magnitude = abs(val)
    if magnitude < 1.0:
        return f"{val:.6g}"
    if magnitude >= 1e16:
        return f"{val:.0f}"
    whole, _, frac = repr(val).partition(".")
    if len(frac) > 6:
        return f"{val:.6f}".rstrip("0").rstrip(".")
    return whole if frac in ("", "0") else f"{whole}.{frac}"


def _float_column_formatter(column: pd.Series) -> Any:
    """Return one formatter for a float column so that its decimal points line up."""
    texts = [format_full_float(v) for v in column.to_numpy(dtype=np.float64) if math.isfinite(v)]
    if any("e" in text for text in texts):
        return format_full_float
    decimals = max((len(text.partition(".")[2]) for text in texts), default=0)
    return lambda v: f"{float(v) + 0.0:.{decimals}f}"


def _json_default(obj: Any) -> Any:
    """Convert numpy values, dataclasses, and sets for `json.dumps`; fall back to `str`."""
    if isinstance(obj, np.generic):
        return obj.item()
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if dataclasses.is_dataclass(obj) and not isinstance(obj, type):
        return dataclasses.asdict(obj)
    if isinstance(obj, (set, frozenset)):
        return sorted(obj, key=str)
    return str(obj)


def _compact_json(value: Any) -> str:
    """Single-line JSON text for one value (falls back to `repr` for unserializable objects)."""
    try:
        return json.dumps(value, default=_json_default)
    except (TypeError, ValueError):
        return repr(value)


def describe_data_object(obj: Any) -> str:
    """One-line type and size summary of a generated object."""
    if isinstance(obj, pd.DataFrame):
        return f"pandas.DataFrame, shape=({obj.shape[0]}, {obj.shape[1]})"
    if isinstance(obj, np.ndarray):
        return f"numpy.ndarray, shape={obj.shape}, dtype={obj.dtype}"
    if isinstance(obj, (dict, list, tuple, set, frozenset)):
        return f"{type(obj).__name__}, len={len(obj)}"
    return type(obj).__name__


def render_data_item(name: str, obj: Any) -> str:
    """Render one named object in full: every row, column, element, and cell text, with no `...` truncation."""
    head = f"[{name}] {describe_data_object(obj)}"
    if isinstance(obj, pd.DataFrame):
        if obj.empty:
            return f"{head}\n(no rows) columns: {list(obj.columns)}"
        formatters: dict[Any, Any] = {}
        if obj.columns.is_unique:
            formatters = {
                col: _float_column_formatter(obj[col])
                for col in obj.columns
                if pd.api.types.is_float_dtype(obj[col])
            }
        show_index = not isinstance(obj.index, pd.RangeIndex)
        if show_index and not isinstance(obj.index, pd.MultiIndex) and pd.api.types.is_float_dtype(obj.index):
            formatters["__index__"] = format_full_float
        table = obj.to_string(
            index=show_index,
            formatters=formatters or None,
            float_format=format_full_float,
        )
        return f"{head}\n{table}"
    if isinstance(obj, np.ndarray):
        body = np.array2string(
            obj,
            threshold=obj.size + 1,
            max_line_width=160,
            formatter={"float_kind": format_full_float},
        )
        return f"{head}\n{body}"
    if isinstance(obj, dict):
        if not obj:
            return f"{head}\n{{}}"
        if all(isinstance(v, pd.DataFrame) for v in obj.values()):
            note = "registry of DataFrames; each table is printed in full by its own section"
            rows = [f"  {key}: shape=({tbl.shape[0]}, {tbl.shape[1]})" for key, tbl in obj.items()]
            return f"{head} ({note})\n" + "\n".join(rows)
        rows = [f"  {json.dumps(str(key))}: {_compact_json(value)}" for key, value in obj.items()]
        return f"{head}\n{{\n" + ",\n".join(rows) + "\n}"
    if isinstance(obj, (list, tuple)):
        if all(v is None or isinstance(v, (str, int, float, bool)) for v in obj):
            return f"{head}\n{_compact_json(list(obj))}"
        return f"{head} (one JSON value per line)\n" + "\n".join(_compact_json(v) for v in obj)
    return f"{head} = {obj!r}"


def print_section_data(section_title: str, items: list[tuple[str, Any]]) -> None:
    """Print everything one section generated, in full, stamped with the date and time of the printout.

    Each item is `(name, obj)`. The whole text passes through `redact_sensitive` before printing, so a
    session token can never reach a notebook output.
    """
    printed_at = datetime.datetime.now().astimezone().strftime("%Y-%m-%d %H:%M:%S %z")
    rule = "=" * 100
    header = f"{rule}\n{section_title}\nPrinted: {printed_at} | {len(items)} item(s)\n{rule}"
    blocks = [render_data_item(name, obj) for name, obj in items]
    print(redact_sensitive(header + "\n" + ("\n" + "-" * 100 + "\n").join(blocks)))

### 2.2 Workspace Paths & Shared Constants

Locates the repository and the database to read. `ACTIVE_DATA_ROOT` is the working directory when it contains its own `state/spaceinvasion.db` (isolated test fixtures); otherwise it is the repository root. `METADATA_DIR` is the folder of the static metadata, and `CONFIG_RESULT_FILE_NAME` names the file that Section 5 writes into it.

In [ ]:
def resolve_repo_root() -> Path:
    """Return the directory that contains the `spaceinvasion` package, searching outward from the working directory."""
    cwd = Path.cwd().resolve()
    for candidate in (cwd, *cwd.parents):
        if (candidate / "spaceinvasion" / "__init__.py").is_file():
            return candidate
    for entry in sys.path:
        if entry and (Path(entry).resolve() / "spaceinvasion" / "__init__.py").is_file():
            return Path(entry).resolve()
    for candidate in Path(game_data.__file__).resolve().parents:
        if (candidate / "spaceinvasion" / "__init__.py").is_file():
            return candidate
    raise RuntimeError("Cannot locate the repository root: no directory contains spaceinvasion/__init__.py")


def resolve_active_data_root(repo_root: Path) -> Path:
    """Prefer the working directory when it has its own `state/spaceinvasion.db`; otherwise use the repository."""
    cwd = Path.cwd().resolve()
    return cwd if (cwd / "state" / "spaceinvasion.db").is_file() else repo_root


REPO_ROOT: Path = resolve_repo_root()
ACTIVE_DATA_ROOT: Path = resolve_active_data_root(REPO_ROOT)
LIVE_DB_PATH: Path = ACTIVE_DATA_ROOT / "state" / "spaceinvasion.db"
METADATA_DIR: Path = REPO_ROOT / "notebooks" / "metadata"
RESOURCE_KEYS: tuple[str, ...] = ("pig_iron", "metal", "kryptonite", "spice")

# Name of the file that Section 5 writes into `METADATA_DIR`: the result of this notebook.
CONFIG_RESULT_FILE_NAME: str = "config_optimizer_result.json"

print_section_data(
    "Section 2.2 data: workspace paths and shared constants",
    [
        ("REPO_ROOT", REPO_ROOT),
        ("ACTIVE_DATA_ROOT", ACTIVE_DATA_ROOT),
        ("LIVE_DB_PATH", LIVE_DB_PATH),
        ("METADATA_DIR", METADATA_DIR),
        ("RESOURCE_KEYS", RESOURCE_KEYS),
        ("CONFIG_RESULT_FILE_NAME", CONFIG_RESULT_FILE_NAME),
    ],
)

### 2.3 Game Catalog Helpers & Formulas

`fmt_duration` turns seconds into text. The other helpers read `game_data` in a fixed order (`research_catalog`, `unit_keys`), fill in the resources that a cost leaves out (`full_cost`), and repair the build times that the wiki truncates at very high levels (`build_monotonic_base_times`). The game's time formulas are here too:

- buildings: `base_time / ((b7 + 1) * 2^b17 * speed)`
- research: `base_time / ((b8 + 1) * speed)`
- ships and defenses: `base_time / (max(1, b9) * 2^b17 * speed)`

where `b7` is the Development center, `b8` the Research lab, `b9` the Weapons factory, `b17` the Microsystem accelerator and `speed` the universe speed factor.

`warehouse_capacity` is the storage formula `100,000 * 2^level`, and `solar_cell_energy_per_unit` reads the catalog's Solar cell temperature table.

In [ ]:
def fmt_duration(seconds: float) -> str:
    """Format seconds as `45s`, `3m 07s`, `5h 35m`, or `12d 04h`."""
    if not math.isfinite(seconds) or seconds <= 0.0:
        return "0s"
    sec_i = int(round(seconds))
    if sec_i < 60:
        return f"{sec_i}s"
    mins, rem_s = divmod(sec_i, 60)
    if mins < 60:
        return f"{mins}m {rem_s:02d}s"
    hrs, rem_m = divmod(mins, 60)
    if hrs < 48:
        return f"{hrs}h {rem_m:02d}m"
    days, rem_h = divmod(hrs, 24)
    return f"{days}d {rem_h:02d}h"


def research_catalog() -> list[tuple[str, dict[str, Any]]]:
    """Researches as `(key, info)` in numeric key order (`r1`, `r2`, ...); wiki entries without an id are left out."""
    return sorted(game_data.RESEARCH_BY_ID.items(), key=lambda entry: int(entry[0][1:]))


def unit_keys(slug_by_key: dict[str, str]) -> list[str]:
    """Ship or defense keys (`s1`, `d1`, ...) in numeric order."""
    return sorted(slug_by_key, key=lambda key: int(key[1:]))


def full_cost(raw_cost: Any) -> dict[str, Any]:
    """Copy a cost mapping, adding a zero for any of the four resources it leaves out."""
    cost = dict(raw_cost or {})
    for resource in RESOURCE_KEYS:
        cost.setdefault(resource, 0)
    return cost


def build_monotonic_base_times(levels_map: dict[int, dict[str, Any]], max_level: int) -> dict[int, float]:
    """Return `{level: base_time_s}`, repairing levels where the wiki's scientific notation lost its exponent.

    At very high levels the scraped time drops to a bare mantissa (for example 1.37 s after 9e13 s). Such a
    level is rebuilt from the previous one, scaled by the same ratio as its Spice-equivalent cost.
    """
    times: dict[int, float] = {}
    previous_time = 0.0
    previous_cost = 0.0
    for level in range(1, max_level + 1):
        row = levels_map.get(level) or {}
        raw_time = float(row.get("base_time_s") or 0.0)
        cost = float(game_data.spice_equivalent(row.get("cost") or {}))
        if level > 1 and previous_time > 0.0 and raw_time < previous_time:
            ratio = (cost / previous_cost) if previous_cost > 0.0 else 1.51
            fixed_time = previous_time * max(1.05, ratio)
        else:
            fixed_time = max(0.0, raw_time)
        times[level] = fixed_time
        previous_time = fixed_time
        previous_cost = cost
    return times


def building_time_divisor(dev_center_level: int, microsystem_level: int, speedup_factor: float) -> float:
    """Divisor of a building's base time: `(b7 + 1) * 2^b17 * speed`, never below 1."""
    return max(1.0, float(dev_center_level + 1) * float(2 ** min(microsystem_level, 40)) * float(speedup_factor))


def research_time_divisor(research_lab_level: int, speedup_factor: float) -> float:
    """Divisor of a research's base time: `(b8 + 1) * speed`, never below 1."""
    return max(1.0, float(research_lab_level + 1) * float(speedup_factor))


def shipyard_time_divisor(weapons_factory_level: int, microsystem_level: int, speedup_factor: float) -> float:
    """Divisor of a ship's or defense's base time: `max(1, b9) * 2^b17 * speed`, never below 1."""
    return max(
        1.0,
        float(max(1, weapons_factory_level)) * float(2 ** min(microsystem_level, 40)) * float(speedup_factor),
    )


def warehouse_capacity(level: int) -> int:
    """Storage capacity of a warehouse at `level`: `100,000 * 2^level` (level 0 is every planet's base capacity)."""
    return int(100_000 * (2 ** max(0, int(level))))


def solar_cell_energy_per_unit(temperature_c: float) -> float:
    """Base energy of one orbital Solar cell (`s9`), interpolated from the catalog's temperature table."""
    energy_by_temperature = game_data.SHIPS[SHIP_SLUG_BY_KEY["s9"]]["solar_cell_energy_by_temp"]
    return float(game_data.interpolate_temp_value(energy_by_temperature, temperature_c))

### 2.4 Read-Only Database Access & JSON Parsing

`open_readonly_db_copy` is the only way this notebook touches SQLite. It copies the database (with its WAL sidecars) into a temporary directory, lets `GameDatabase` bring the copy to the current schema, and yields a query-only connection to the copy. The live file is never opened, locked or modified.

`read_latest_game_state` returns the bot's latest full game state, which the loaders of Section 2.6 read. `parse_json_object` and `parse_level_map` decode the JSON text columns.

In [ ]:
def parse_json_object(raw: Any) -> dict[str, Any]:
    """Return `raw` as a dict when it is a dict or the JSON text of an object; otherwise `{}`."""
    if isinstance(raw, dict):
        return dict(raw)
    if isinstance(raw, str) and raw.strip():
        try:
            parsed = json.loads(raw)
        except ValueError:
            return {}
        return dict(parsed) if isinstance(parsed, dict) else {}
    return {}


def parse_level_map(raw: Any) -> dict[str, int]:
    """Parse a `{key: level or count}` object into non-negative ints, dropping non-numeric values."""
    return {
        str(key): max(0, int(value or 0))
        for key, value in parse_json_object(raw).items()
        if isinstance(value, (int, float))
    }


def has_database(db_path: Path) -> bool:
    """True when `db_path` is a file large enough to hold a SQLite header."""
    return db_path.is_file() and db_path.stat().st_size >= 100


@contextlib.contextmanager
def open_readonly_db_copy(db_path: Path) -> Iterator[sqlite3.Connection]:
    """Yield a query-only connection to a temporary copy of `db_path`, migrated to the current schema.

    The live database is only copied (with its WAL sidecars), never opened, so the notebook cannot lock
    or change it. Opening the copy once through `GameDatabase` applies any pending forward-only
    migrations, which lets the notebook's SQL rely on the current schema.
    """
    with tempfile.TemporaryDirectory(prefix="si_fp_ro_") as tmp_dir:
        tmp_db = Path(tmp_dir) / db_path.name
        shutil.copy2(db_path, tmp_db)
        tmp_db.chmod(0o600)
        for suffix in ("-wal", "-shm", "-journal"):
            sidecar = Path(f"{db_path}{suffix}")
            if sidecar.is_file():
                tmp_sidecar = Path(f"{tmp_db}{suffix}")
                shutil.copy2(sidecar, tmp_sidecar)
                tmp_sidecar.chmod(0o600)
        GameDatabase(tmp_db, backup_on_upgrade=False).close()
        with contextlib.closing(sqlite3.connect(tmp_db)) as conn:
            conn.row_factory = sqlite3.Row
            conn.execute("PRAGMA query_only = ON")
            yield conn


SQL_LATEST_GAME_STATE_META = """
SELECT value
FROM meta
WHERE key = 'latest_game_state'
LIMIT 1
"""


def read_latest_game_state(conn: sqlite3.Connection) -> dict[str, Any]:
    """Return the bot's most recent full game state (`meta['latest_game_state']`), or `{}` when it is absent."""
    row = conn.execute(SQL_LATEST_GAME_STATE_META).fetchone()
    return parse_json_object(row[0]) if row and row[0] else {}

### 2.5 Static Metadata (`notebooks/metadata/*.json`)

Static data is kept in JSON files instead of code cells. This notebook reads three of them:

| File | Content |
|---|---|
| `technologies.json` | domain, effect, per-level bonus and scope of `r1`–`r15` |
| `resources.json` | mine and warehouse building of each resource, Mining technology flag, notes |
| `planner_settings.json` | planner settings and fallback assumptions: empire size, generic planet, cold-start colony |

Loading stops with an error when a file misses an entity, has an unknown one, lacks a field, or names an entity differently from `game_data`, so a description can never end up on the wrong entity.

`technology_multiplier` turns a research level into its multiplier, using the `per_level_bonus_pct` of `technologies.json`.

In [ ]:
def load_metadata(file_name: str) -> dict[str, Any]:
    """Read one JSON document from `notebooks/metadata/`."""
    return json.loads((METADATA_DIR / file_name).read_text(encoding="utf-8"))


def load_keyed_metadata(
    file_name: str,
    section: str,
    expected_keys: Iterable[str],
    required_fields: tuple[str, ...],
) -> dict[str, dict[str, Any]]:
    """Load `{key: entry}` metadata; it must cover exactly `expected_keys`, each entry with every required field."""
    entries = load_metadata(file_name)[section]
    expected = set(expected_keys)
    if set(entries) != expected:
        raise ValueError(
            f"{file_name}: missing {sorted(expected - set(entries))}, unknown {sorted(set(entries) - expected)}"
        )
    for key, entry in entries.items():
        absent = [field for field in required_fields if field not in entry]
        if absent:
            raise ValueError(f"{file_name}: {key} lacks {absent}")
    return entries


def load_catalog_metadata(
    file_name: str,
    section: str,
    catalog_names: dict[str, str],
    required_fields: tuple[str, ...],
) -> dict[str, dict[str, Any]]:
    """Load per-entity metadata and check each entry's `name` against the game catalog (`{key: name}`)."""
    entries = load_keyed_metadata(file_name, section, catalog_names, ("name", *required_fields))
    for key, entry in entries.items():
        if entry["name"] != catalog_names[key]:
            raise ValueError(
                f"{file_name}: {key} is named {entry['name']!r}, but game_data calls it {catalog_names[key]!r}"
            )
    return entries


TECHNOLOGY_METADATA: dict[str, dict[str, Any]] = load_catalog_metadata(
    "technologies.json",
    "technologies",
    {key: str(info["name"]) for key, info in research_catalog()},
    ("domain", "effect_metric", "per_level_bonus_pct", "applies_to", "strategic_effect"),
)

RESOURCE_METADATA: dict[str, dict[str, Any]] = load_keyed_metadata(
    "resources.json",
    "resources",
    RESOURCE_KEYS,
    ("mine_building_key", "warehouse_building_key", "boosted_by_mining_tech", "notes"),
)
PLANNER_SETTINGS: dict[str, Any] = load_metadata("planner_settings.json")


def technology_multiplier(research_key: str, level: int) -> float:
    """Multiplier a research gives at `level`: `1 + per_level_bonus_pct / 100 * level` (1.0 for an unknown key)."""
    bonus_per_level = float(TECHNOLOGY_METADATA.get(research_key, {}).get("per_level_bonus_pct", 0.0)) / 100.0
    return 1.0 + bonus_per_level * float(level)


print_section_data(
    "Section 2.5 data: static metadata",
    [
        ("TECHNOLOGY_METADATA", TECHNOLOGY_METADATA),
        ("RESOURCE_METADATA", RESOURCE_METADATA),
        ("PLANNER_SETTINGS", PLANNER_SETTINGS),
    ],
)

### 2.6 Planet Roster & Account Helpers

`load_live_planet_states` reads every owned planet from the temporary database copy. It prefers the bot's latest full game state, which records the real warehouse capacities. Without one, it takes the newest `planet_snapshots` row of each planet and derives the capacities from the warehouse levels.

`build_padded_empire_planets` appends generic planets to the owned ones until the empire has the given number of normal planets. Asteroids are kept but do not count toward that limit. A generic planet (`empty_colony_slot`) is an empty colony slot: a planet with the given number of free fields, no resources, level-0 warehouses and nothing built. Its `planet_id` is negative (`-1`, `-2`, …) and its coordinates are `EMPTY:01`, `EMPTY:02`, ….

`planet_param_context` joins the account-wide inputs with one planet's own values: its temperature and the levels of the buildings that shorten build times. `planet_identity` picks the four values that identify a planet in a table.

`load_account_snapshot` reads what is the same on every planet: your research levels, from the newest planet snapshot, and the universe speed, from the bot's latest game state.

In [ ]:
SQL_LATEST_PLANETS_PER_ID = """
SELECT
    p.planet_id,
    p.coords,
    p.name,
    p.planet_type,
    p.temperature,
    p.fields_used,
    p.fields_max,
    p.pig_iron,
    p.metal,
    p.kryptonite,
    p.spice,
    p.energy_produced,
    p.energy_used,
    p.buildings_json,
    p.ships_json,
    p.defense_json,
    p.economy_rates_json
FROM planet_snapshots p
INNER JOIN (
    SELECT planet_id, MAX(id) AS max_id
    FROM planet_snapshots
    GROUP BY planet_id
) latest ON latest.max_id = p.id
ORDER BY p.planet_id ASC
"""


def snapshot_row_to_planet_fields(row: sqlite3.Row) -> dict[str, Any]:
    """Reshape a `planet_snapshots` row into the nested layout the saved game state uses for a planet."""
    return {
        "planet_id": row["planet_id"],
        "name": row["name"],
        "coords": row["coords"],
        "planet_type": row["planet_type"],
        "temperature": row["temperature"],
        "fields_used": row["fields_used"],
        "fields_max": row["fields_max"],
        "resources": {resource: row[resource] for resource in RESOURCE_KEYS},
        "energy": {"produced": row["energy_produced"], "used": row["energy_used"]},
        "buildings": row["buildings_json"],
        "ships": row["ships_json"],
        "defense": row["defense_json"],
        "economy_rates": row["economy_rates_json"],
    }


def planet_state_from_fields(fields: dict[str, Any]) -> PlanetState:
    """Build a typed `PlanetState` from one planet's saved fields.

    Gaps are filled from the building levels: used fields default to the sum of all levels, and a missing
    warehouse capacity to `warehouse_capacity(level)`. A missing size defaults to `fallback_planet_fields`.
    """
    buildings = parse_level_map(fields.get("buildings"))
    resources = parse_json_object(fields.get("resources"))
    storage = parse_json_object(fields.get("storage"))
    energy = parse_json_object(fields.get("energy"))
    fields_used = int(fields.get("fields_used") or sum(buildings.values()))
    fallback_fields = int(PLANNER_SETTINGS["empire"]["fallback_planet_fields"])
    return PlanetState(
        planet_id=int(fields.get("planet_id") or 0),
        name=str(fields.get("name") or "Colony"),
        coords=str(fields.get("coords") or "00:000:00"),
        planet_type=str(fields.get("planet_type") or "normal"),
        temperature=int(fields.get("temperature") or 0),
        fields_used=fields_used,
        fields_max=int(fields.get("fields_max") or max(fallback_fields, fields_used)),
        resources=Resources(**{resource: int(resources.get(resource) or 0) for resource in RESOURCE_KEYS}),
        storage=Resources(
            **{
                resource: int(
                    storage.get(resource)
                    or warehouse_capacity(buildings.get(RESOURCE_METADATA[resource]["warehouse_building_key"], 0))
                )
                for resource in RESOURCE_KEYS
            }
        ),
        energy=EnergyBalance(produced=int(energy.get("produced") or 0), used=int(energy.get("used") or 0)),
        buildings=buildings,
        ships=parse_level_map(fields.get("ships")),
        defense=parse_level_map(fields.get("defense")),
        economy_rates=parse_level_map(fields.get("economy_rates")),
    )


def load_live_planet_states(db_path: Path) -> list[PlanetState]:
    """Load every owned planet from a temporary copy of `db_path`.

    The bot's latest full game state is preferred because it records the real warehouse capacities. Without
    it, the newest `planet_snapshots` row of each planet is used and capacities come from the warehouse levels.
    """
    if not has_database(db_path):
        return []
    with open_readonly_db_copy(db_path) as conn:
        saved_planets = read_latest_game_state(conn).get("planets") or []
        if isinstance(saved_planets, list) and saved_planets:
            return [planet_state_from_fields(planet) for planet in saved_planets if isinstance(planet, dict)]
        return [
            planet_state_from_fields(snapshot_row_to_planet_fields(row))
            for row in conn.execute(SQL_LATEST_PLANETS_PER_ID)
        ]


def empty_colony_slot(slot_number: int, fields: int, temperature_c: int) -> PlanetState:
    """A generic planet that is not colonized yet: `fields` free fields, base warehouses, nothing built."""
    base_storage = warehouse_capacity(0)
    return PlanetState(
        planet_id=-slot_number,
        name=f"Empty Colony Slot {slot_number}",
        coords=f"EMPTY:{slot_number:02d}",
        planet_type="normal",
        temperature=int(temperature_c),
        fields_used=0,
        fields_max=int(fields),
        resources=Resources(0, 0, 0, 0),
        storage=Resources(base_storage, base_storage, base_storage, base_storage),
        energy=EnergyBalance(0, 0),
        buildings={f"b{building_id}": 0 for building_id in sorted(game_data.BUILDING_BY_ID)},
        ships={},
        defense={},
        economy_rates={},
    )


def build_padded_empire_planets(
    live_planets: list[PlanetState],
    *,
    max_planets: int,
    empty_planet_size: int,
    empty_temp_c: int,
) -> list[PlanetState]:
    """Return the owned planets followed by generic planets (empty colony slots).

    Slots are added until the empire has `max_planets` normal planets; asteroids and other planet types are
    kept but do not count toward that limit.
    """
    owned_normal_planets = sum(1 for planet in live_planets if planet.planet_type == "normal")
    return [
        *live_planets,
        *(
            empty_colony_slot(slot_number, empty_planet_size, empty_temp_c)
            for slot_number in range(1, max(0, int(max_planets) - owned_normal_planets) + 1)
        ),
    ]


def planet_param_context(
    account_context: dict[str, Any],
    slot_index: int,
    planet: PlanetState,
    *,
    is_empty_slot: bool,
) -> dict[str, Any]:
    """Inputs of the table builders for one planet: the account-wide context plus the planet's own values."""
    return {
        **account_context,
        "slot_index": slot_index,
        "planet_id": int(planet.planet_id),
        "coords": planet.coords,
        "planet_status": "empty_colony_slot" if is_empty_slot else "colonized",
        "planet_name": planet.name,
        "planet_type": planet.planet_type,
        "planet_temperature_c": int(planet.temperature),
        "b7_dev_center": planet.buildings.get("b7", 0),
        "b8_research_lab": planet.buildings.get("b8", 0),
        "b9_weapons_factory": planet.buildings.get("b9", 0),
        "b17_microsystem": planet.buildings.get("b17", 0),
    }


def planet_identity(context: dict[str, Any]) -> dict[str, Any]:
    """The four columns that identify a planet in every per-planet table."""
    return {column: context[column] for column in ("slot_index", "planet_id", "coords", "planet_status")}


SQL_LATEST_COLONY_SNAPSHOT = """
SELECT coords, buildings_json, researches_json
FROM planet_snapshots
ORDER BY ts DESC, id DESC
LIMIT 1
"""


def load_account_snapshot(db_path: Path) -> dict[str, Any]:
    """Your own research levels and the universe speed, read from a temporary copy of `db_path`.

    The research levels come from the newest planet snapshot, the speed from the bot's latest game state.
    Without a database, or without those records, the research is empty and the speed is the one of
    `cold_start_colony` in `planner_settings.json`.
    """
    colony_researches: dict[str, int] = {}
    universe_speedup_factor = float(PLANNER_SETTINGS["cold_start_colony"]["speedup_factor"])
    if has_database(db_path):
        with open_readonly_db_copy(db_path) as conn:
            snapshot = conn.execute(SQL_LATEST_COLONY_SNAPSHOT).fetchone()
            if snapshot:
                colony_researches = parse_level_map(snapshot["researches_json"])
            saved_speed = read_latest_game_state(conn).get("speedup_factor")
            if isinstance(saved_speed, (int, float)) and math.isfinite(float(saved_speed)) and float(saved_speed) > 0.0:
                universe_speedup_factor = float(saved_speed)
    return {"colony_researches": colony_researches, "universe_speedup_factor": universe_speedup_factor}

### 2.7 Building Economy

`building_level_economy` returns what one level of a building does per hour on a given planet: its output, the Pig iron or Spice that it uses as feedstock, the value of both in Spice-equivalent, and the energy that it produces or uses. The universe speed, Mining technology and Energy technology are applied; the Blast furnace is not boosted by Mining technology, and the Spice mine follows the planet's temperature. `produced_resource` names the resource that a building produces.

The production model of Section 4.2 is built from these numbers.

In [ ]:
def produced_resource(building_key: str) -> str:
    """Resource that a building produces (`"none"` for a building that is not a mine)."""
    return next(
        (resource for resource, meta in RESOURCE_METADATA.items() if meta["mine_building_key"] == building_key),
        "none",
    )


def building_level_economy(building_id: int, level: int, context: dict[str, Any]) -> dict[str, float]:
    """Hourly output, feedstock, net Spice-equivalent value and energy of one building level on the context's planet.

    Level 0, a building that is not built, yields zeros.
    """
    row = (game_data.BUILDING_BY_ID[building_id].get("levels") or {}).get(level) or {}
    merchant_rates = game_data.MERCHANT_RATES_IN_SPICE
    resource = produced_resource(f"b{building_id}")
    speedup = float(context["speedup_factor"])

    # Hourly output scales with universe speed; Mining technology adds +2 % per level (not to the Blast furnace).
    # Only the Spice mine depends on the planet: its output follows the temperature.
    if resource != "none":
        temperature = int(context["planet_temperature_c"])
        production_1x = float(
            game_data.mine_production_per_hour(building_id, level, temperature=temperature, mining_tech_level=0)
        )
        production = speedup * float(
            game_data.mine_production_per_hour(
                building_id, level, temperature=temperature, mining_tech_level=int(context["r1_mining_tech"])
            )
        )
    else:
        production_1x = 0.0
        production = 0.0
    # Feedstock per hour: the Blast furnace's Pig iron (2 per Metal) scales with universe speed like its output;
    # the Thermosolar plant's Spice fuel is the catalog value, not multiplied by universe speed.
    pig_iron_used = float(row.get("pig_iron_used") or 0.0) * speedup
    spice_used = float(row.get("spice_used") or 0.0)
    gross_spice_eq_hr = production / float(merchant_rates[resource]) if resource in merchant_rates else 0.0
    feedstock_spice_eq_hr = pig_iron_used / float(merchant_rates["pig_iron"]) + spice_used / float(
        merchant_rates["spice"]
    )
    # Energy: Energy technology boosts production (positive values) only; consumption is unchanged.
    energy_base = float(row.get("energy") or 0.0)
    energy_multiplier = technology_multiplier("r2", context["r2_energy_tech"])
    return {
        "production_1x": production_1x,
        "production": production,
        "pig_iron_used": pig_iron_used,
        "spice_used": spice_used,
        "net_spice_eq_hr": gross_spice_eq_hr - feedstock_spice_eq_hr,
        "energy_base": energy_base,
        "energy": (energy_base * energy_multiplier) if energy_base > 0.0 else energy_base,
    }

### 2.8 Technology Requirement Graph

`TechnologyRequirementGraph` holds the prerequisites of every building, research, ship and defense as a directed graph, built once from `game_data`. A node is an entity key (`b7`, `r2`, `s16`, `d10`). An edge `source -> target` with a level means that the target needs the source at that level. For example, the Weapons factory `b9` needs the Development center `b7` at level 3 and Energy technology `r2` at level 1.

Section 4.1 asks the graph which buildings and research levels the plan requires: `prereqs_of[key]` lists the direct requirements of an entity, and `transitive_requirements` the minimum level of every direct and indirect prerequisite. `TECH_GRAPH` is the graph of the catalog.

In [ ]:
class TechnologyRequirementGraph:
    """In-memory directed acyclic graph of the prerequisites of every building, research, ship and defense.

    Nodes are entity keys (`b7`, `r2`, `s7`, `d10`). An edge `source -> target` with a level means that the
    target needs the source at that level. Ships and defenses have no levels of their own.
    """

    PLAN_COLUMNS: tuple[str, ...] = (
        "step",
        "key",
        "name",
        "category",
        "topological_layer",
        "from_level",
        "to_level",
        "levels_added",
        "pig_iron",
        "metal",
        "kryptonite",
        "spice",
        "step_cost_spice_eq",
        "step_effective_time_s",
        "step_effective_time_text",
        "cum_cost_spice_eq",
        "cum_effective_time_s",
        "cum_effective_time_text",
    )

    def __init__(self) -> None:
        self.nodes: dict[str, dict[str, Any]] = {}
        self._requirements: dict[str, list[dict[str, Any]]] = {}
        self._base_times: dict[str, dict[int, float]] = {}
        for building_id in sorted(game_data.BUILDING_BY_ID):
            self._add_node(f"b{building_id}", "buildings", game_data.BUILDING_BY_ID[building_id])
        for key, info in research_catalog():
            self._add_node(key, "research", info)
        for key in unit_keys(SHIP_SLUG_BY_KEY):
            self._add_node(key, "ships", game_data.SHIPS[SHIP_SLUG_BY_KEY[key]])
        for key in unit_keys(DEFENSE_SLUG_BY_KEY):
            self._add_node(key, "defense", game_data.DEFENSES[DEFENSE_SLUG_BY_KEY[key]])

        self.prereqs_of, self.unlocks_of = self._link_requirements()
        self.topological_order, self.topological_layer = self._sort_topologically()
        self._topological_rank = {key: rank for rank, key in enumerate(self.topological_order)}
        self.ancestors_of, self.descendants_of = self._close_transitively()

    def __repr__(self) -> str:
        edges = sum(len(requirements) for requirements in self.prereqs_of.values())
        return f"TechnologyRequirementGraph(nodes={len(self.nodes)}, edges={edges})"

    def _add_node(self, key: str, category: str, info: dict[str, Any]) -> None:
        """Register one catalog entity with its requirements and its repaired base times."""
        levels = info.get("levels") or {}
        max_level = int(info.get("max_table_level") or len(levels))
        self.nodes[key] = {
            "key": key,
            "category": category,
            "entity_id": int(key[1:]),
            "slug": str(info["slug"]),
            "name": str(info["name"]),
            "max_level": max_level,
        }
        self._requirements[key] = list(info.get("requirements") or [])
        self._base_times[key] = build_monotonic_base_times(levels, max_level)

    def _link_requirements(
        self,
    ) -> tuple[dict[str, tuple[tuple[str, int], ...]], dict[str, tuple[tuple[str, int], ...]]]:
        """Turn each catalog requirement into an edge, indexed by target (`prereqs_of`) and source (`unlocks_of`)."""
        key_by_slug = {(node["category"], node["slug"]): key for key, node in self.nodes.items()}
        prereqs: dict[str, list[tuple[str, int]]] = {key: [] for key in self.nodes}
        unlocks: dict[str, list[tuple[str, int]]] = {key: [] for key in self.nodes}
        for target, requirements in self._requirements.items():
            for requirement in requirements:
                source = key_by_slug.get((str(requirement["category"]), str(requirement["slug"])))
                if source is None:
                    raise KeyError(f"Unresolved tech-tree requirement {requirement!r} on {target}")
                level = int(requirement["level"])
                prereqs[target].append((source, level))
                unlocks[source].append((target, level))
        return (
            {key: tuple(edges) for key, edges in prereqs.items()},
            {key: tuple(edges) for key, edges in unlocks.items()},
        )

    def _sort_topologically(self) -> tuple[tuple[str, ...], dict[str, int]]:
        """Kahn's algorithm: order the nodes so prerequisites come first, and record each node's dependency depth."""
        open_prereqs = {key: len(self.prereqs_of[key]) for key in self.nodes}
        layer = {key: 0 for key in self.nodes}
        queue: deque[str] = deque(key for key, count in open_prereqs.items() if count == 0)
        order: list[str] = []
        while queue:
            node = queue.popleft()
            order.append(node)
            for dependent, _level in self.unlocks_of[node]:
                layer[dependent] = max(layer[dependent], layer[node] + 1)
                open_prereqs[dependent] -= 1
                if open_prereqs[dependent] == 0:
                    queue.append(dependent)
        if len(order) != len(self.nodes):
            raise ValueError(f"Cycle detected in technology requirement graph: {sorted(set(self.nodes) - set(order))}")
        return tuple(order), layer

    def _close_transitively(self) -> tuple[dict[str, frozenset[str]], dict[str, frozenset[str]]]:
        """Collect each node's direct and indirect prerequisites (`ancestors_of`) and dependents (`descendants_of`)."""
        ancestors: dict[str, set[str]] = {key: set() for key in self.nodes}
        for node in self.topological_order:
            for prerequisite, _level in self.prereqs_of[node]:
                ancestors[node].add(prerequisite)
                ancestors[node].update(ancestors[prerequisite])
        descendants: dict[str, set[str]] = {key: set() for key in self.nodes}
        for node in reversed(self.topological_order):
            for dependent, _level in self.unlocks_of[node]:
                descendants[node].add(dependent)
                descendants[node].update(descendants[dependent])
        return (
            {key: frozenset(found) for key, found in ancestors.items()},
            {key: frozenset(found) for key, found in descendants.items()},
        )

    def current_level(self, key: str, buildings: dict[str, int], researches: dict[str, int]) -> int:
        """Level of `key` in the given state (ships and defenses have no level, so they return 0)."""
        category = self.nodes[key]["category"]
        if category == "buildings":
            return max(0, int(buildings.get(key, 0) or 0))
        if category == "research":
            return max(0, int(researches.get(key, 0) or 0))
        return 0

    def direct_missing(
        self,
        target_key: str,
        buildings: dict[str, int],
        researches: dict[str, int],
    ) -> list[dict[str, Any]]:
        """Direct prerequisites of `target_key` that the given state does not meet yet."""
        missing: list[dict[str, Any]] = []
        for source, required_level in self.prereqs_of[target_key]:
            level = self.current_level(source, buildings, researches)
            if level < required_level:
                missing.append(
                    {
                        "key": source,
                        "name": self.nodes[source]["name"],
                        "category": self.nodes[source]["category"],
                        "required_level": required_level,
                        "current_level": level,
                        "missing_levels": required_level - level,
                    }
                )
        return missing

    def is_unlocked(self, target_key: str, buildings: dict[str, int], researches: dict[str, int]) -> bool:
        """True when the given state meets every direct prerequisite of `target_key`."""
        return not self.direct_missing(target_key, buildings, researches)

    def transitive_requirements(self, target_key: str, *, include_target_level: int = 0) -> dict[str, int]:
        """Minimum level of every direct and indirect prerequisite of `target_key`, in topological order.

        When several requirement paths reach the same entity, the highest required level wins.
        `include_target_level` also demands that level of the target itself (buildings and researches only).
        """
        if target_key not in self.nodes:
            raise KeyError(f"Unknown node key: {target_key!r}")
        required: dict[str, int] = {}
        if include_target_level > 0 and self.nodes[target_key]["category"] in ("buildings", "research"):
            required[target_key] = int(include_target_level)
        for source, level in self.prereqs_of[target_key]:
            required[source] = max(required.get(source, 0), int(level))
        # Visiting nodes from the last in topological order to the first sees every dependent before its prerequisites.
        for node in reversed(self.topological_order):
            if node not in required:
                continue
            for source, level in self.prereqs_of[node]:
                if level > required.get(source, 0):
                    required[source] = int(level)
        return {key: required[key] for key in sorted(required, key=self._topological_rank.__getitem__)}

    def _level_cost_and_base_time(self, key: str, level: int) -> tuple[dict[str, Any], float]:
        """Cost and base time of taking building or research `key` to `level`."""
        node = self.nodes[key]
        base_time = float(self._base_times[key].get(level, 0.0))
        if node["category"] == "buildings":
            return game_data.building_cost(int(node["entity_id"]), level), base_time
        levels = game_data.RESEARCHES[node["slug"]].get("levels") or {}
        return full_cost((levels.get(level) or {}).get("cost")), base_time

    def plan_unlock_sequence(
        self,
        target_key: str,
        buildings: dict[str, int],
        researches: dict[str, int],
        *,
        speedup_factor: float,
        include_target_level: int = 0,
    ) -> pd.DataFrame:
        """Ordered upgrades that take the given state to the point where `target_key` is unlocked.

        One row per building or research that must be raised, in an order that respects every prerequisite.
        Times use the simulated state, so a Development center, Research lab or Microsystem accelerator
        raised by an earlier step shortens the later ones.
        """
        simulated_buildings = {str(key): max(0, int(level or 0)) for key, level in buildings.items()}
        simulated_researches = {str(key): max(0, int(level or 0)) for key, level in researches.items()}
        speed = max(0.1, float(speedup_factor))

        rows: list[dict[str, Any]] = []
        total_spice_eq = 0.0
        total_time = 0.0
        requirements = self.transitive_requirements(target_key, include_target_level=include_target_level)
        for key, required_level in requirements.items():
            node = self.nodes[key]
            current = self.current_level(key, simulated_buildings, simulated_researches)
            if current >= required_level:
                continue
            step_cost = {resource: 0 for resource in RESOURCE_KEYS}
            step_time = 0.0
            for level in range(current + 1, required_level + 1):
                cost, base_time = self._level_cost_and_base_time(key, level)
                for resource in RESOURCE_KEYS:
                    step_cost[resource] += int(cost.get(resource, 0))
                if node["category"] == "buildings":
                    divisor = building_time_divisor(
                        simulated_buildings.get("b7", 0), simulated_buildings.get("b17", 0), speed
                    )
                    simulated_buildings[key] = level
                else:
                    divisor = research_time_divisor(simulated_buildings.get("b8", 0), speed)
                    simulated_researches[key] = level
                step_time += base_time / divisor

            step_spice_eq = float(game_data.spice_equivalent(step_cost))
            total_spice_eq += step_spice_eq
            total_time += step_time
            rows.append(
                {
                    "step": len(rows) + 1,
                    "key": key,
                    "name": node["name"],
                    "category": node["category"],
                    "topological_layer": self.topological_layer[key],
                    "from_level": current,
                    "to_level": required_level,
                    "levels_added": required_level - current,
                    "pig_iron": step_cost["pig_iron"],
                    "metal": step_cost["metal"],
                    "kryptonite": step_cost["kryptonite"],
                    "spice": step_cost["spice"],
                    "step_cost_spice_eq": round(step_spice_eq, 2),
                    "step_effective_time_s": round(step_time, 2),
                    "step_effective_time_text": fmt_duration(step_time),
                    "cum_cost_spice_eq": round(total_spice_eq, 2),
                    "cum_effective_time_s": round(total_time, 2),
                    "cum_effective_time_text": fmt_duration(total_time),
                }
            )
        return pd.DataFrame(rows, columns=list(self.PLAN_COLUMNS))


TECH_GRAPH = TechnologyRequirementGraph()

print_section_data(
    "Section 2.8 data: technology requirement graph",
    [
        (
            "TECH_GRAPH (TechnologyRequirementGraph summary)",
            {
                "nodes": len(TECH_GRAPH.nodes),
                "edges": sum(len(requirements) for requirements in TECH_GRAPH.prereqs_of.values()),
                "max_topological_layer": max(TECH_GRAPH.topological_layer.values()),
            },
        ),
    ],
)

### 2.9 Universe Research Helpers

The bot records what it learns about the research levels of other players in the `universe_research_max` table of its database. `load_universe_research_sources` reads that ledger and the researches listed in the archived spy reports from the temporary database copy (`research_key_aliases` maps the names that reports use to research keys). `build_universe_research_table` combines both with your own levels into one row per research (`universe_research_row`).

In [ ]:
SQL_UNIVERSE_RESEARCH_LEDGER = """
SELECT
    research_key,
    research_id,
    slug,
    name,
    own_level,
    spied_max_level,
    server_record_level,
    universe_max_level,
    holder,
    holder_coords,
    updated_ts
FROM universe_research_max
ORDER BY research_id ASC, slug ASC
"""

SQL_SPY_REPORT_RESEARCHES = """
SELECT target_coords, owner, researches_json
FROM spy_reports
WHERE researches_json != '' AND researches_json != '{}'
ORDER BY ts ASC, id ASC
"""


def research_key_aliases() -> dict[str, str]:
    """Key, slug and name of every catalog research in lower case, each mapped to the key (`r1`, `r2`, ...)."""
    aliases: dict[str, str] = {}
    for key, info in research_catalog():
        for alias in (key, str(info["slug"]), str(info["name"])):
            aliases[alias.lower()] = key
    return aliases


def load_universe_research_sources(db_path: Path) -> dict[str, Any]:
    """Read what the bot recorded about research levels in the universe from a temporary copy of `db_path`.

    `ledger_rows` are the rows of the bot's `universe_research_max` table. `spied_levels` holds, per research,
    the highest level found in the archived spy reports and the planet it was seen on. Both stay empty without
    a database.
    """
    sources: dict[str, Any] = {"ledger_rows": [], "spied_levels": {}, "spy_reports_with_research": 0}
    if not has_database(db_path):
        return sources
    aliases = research_key_aliases()
    spied_levels: dict[str, dict[str, Any]] = sources["spied_levels"]
    with open_readonly_db_copy(db_path) as conn:
        sources["ledger_rows"] = [dict(row) for row in conn.execute(SQL_UNIVERSE_RESEARCH_LEDGER)]
        for report in conn.execute(SQL_SPY_REPORT_RESEARCHES):
            levels = {
                aliases[name.strip().lower()]: level
                for name, level in parse_level_map(report["researches_json"]).items()
                if level > 0 and name.strip().lower() in aliases
            }
            if levels:
                sources["spy_reports_with_research"] += 1
            for key, level in levels.items():
                # Reports are read oldest first, so the newest sighting of the highest level is kept.
                if level >= spied_levels.get(key, {}).get("level", 0):
                    spied_levels[key] = {
                        "level": level,
                        "holder": str(report["owner"] or ""),
                        "holder_coords": str(report["target_coords"] or ""),
                    }
    return sources


def universe_research_row(
    key: str,
    info: dict[str, Any] | None,
    ledger_row: dict[str, Any],
    spied: dict[str, Any],
    own_level: int,
) -> dict[str, Any]:
    """One `universe_research_df` row: the levels each source reports for a research and the maximum of them."""
    levels = (info or {}).get("levels") or {}
    own = max(int(own_level), int(ledger_row.get("own_level") or 0))
    spied_level = max(int(spied.get("level") or 0), int(ledger_row.get("spied_max_level") or 0))
    server_record = int(ledger_row.get("server_record_level") or 0)
    ledger_max = int(ledger_row.get("universe_max_level") or 0)
    universe_max = max(own, spied_level, server_record, ledger_max)

    if universe_max == 0:
        max_source = "none"
    elif server_record == universe_max:
        max_source = "server_record"
    elif spied_level == universe_max:
        max_source = "spy_report"
    elif own == universe_max:
        max_source = "own_level"
    else:  # the ledger never lowers a maximum, so it can remember more than today's sources show
        max_source = "ledger"
    if max_source == "spy_report" and int(spied.get("level") or 0) == universe_max:
        holder, holder_coords = str(spied["holder"]), str(spied["holder_coords"])
    elif max_source != "own_level" and ledger_max == universe_max > 0:
        holder, holder_coords = str(ledger_row.get("holder") or ""), str(ledger_row.get("holder_coords") or "")
    else:
        holder, holder_coords = "", ""

    updated_ts = float(ledger_row.get("updated_ts") or 0.0)
    return {
        "research_id": int(key[1:]) if info is not None else int(ledger_row.get("research_id") or 0),
        "key": key,
        "slug": str(info["slug"]) if info is not None else str(ledger_row.get("slug") or key),
        "name": str(info["name"]) if info is not None else str(ledger_row.get("name") or key),
        "own_level": own,
        "spied_max_level": spied_level,
        "server_record_level": server_record,
        "universe_max_level": universe_max,
        "max_source": max_source,
        "holder": holder,
        "holder_coords": holder_coords,
        "gap_to_max": universe_max - own,
        "own_percent_of_max": round(100.0 * float(own) / float(universe_max), 1) if universe_max > 0 else 0.0,
        "catalog_max_level": int((info or {}).get("max_table_level") or len(levels)),
        "ledger_updated_utc": (
            datetime.datetime.fromtimestamp(updated_ts, datetime.timezone.utc).strftime("%Y-%m-%d %H:%M:%S")
            if updated_ts > 0.0
            else ""
        ),
    }


def build_universe_research_table(sources: dict[str, Any], own_levels: dict[str, int]) -> pd.DataFrame:
    """One row per research of the catalog, followed by any other research the bot's ledger tracks."""
    catalog = dict(research_catalog())
    ledger = {str(row["research_key"]): row for row in sources["ledger_rows"]}
    spied_levels = sources["spied_levels"]
    return pd.DataFrame(
        [
            universe_research_row(
                key,
                catalog.get(key),
                ledger.get(key, {}),
                spied_levels.get(key, {}),
                max(0, int(own_levels.get(key, 0) or 0)),
            )
            for key in (*catalog, *(key for key in ledger if key not in catalog))
        ]
    )

## 3. Inputs

What the optimizer plans with: your planets and the research levels. Both come from the temporary database copy. Without a database the roster holds generic planets only, the research levels are empty, and the universe speed is the one of `cold_start_colony` in `planner_settings.json`.

### 3.1 Account, Planet Roster and Contexts

- `live_planet_states` — every planet you own;
- `empire_planet_states` — the roster of the plan: the owned planets, followed by generic planets (empty colony slots of `EMPTY_COLONY_SIZE_FIELDS` fields at `EMPTY_COLONY_DEFAULT_TEMP_C` °C) until the empire has `MAX_EMPIRE_PLANETS` normal planets;
- `ACCOUNT_SNAPSHOT`, `colony_researches` — your research levels and the universe speed;
- `GAME_PARAM_CONTEXT` — the account-wide inputs of the formulas: the universe speed, Mining technology and Energy technology;
- `PLANET_PARAM_CONTEXTS` — one context per planet of the roster: the account-wide inputs plus the planet's temperature and the levels of its Development center, Research lab, Weapons factory and Microsystem accelerator.

In [ ]:
MAX_EMPIRE_PLANETS: int = int(PLANNER_SETTINGS["empire"]["max_planets"])
EMPTY_COLONY_SIZE_FIELDS: int = int(PLANNER_SETTINGS["empire"]["empty_colony_fields"])
EMPTY_COLONY_DEFAULT_TEMP_C: int = int(PLANNER_SETTINGS["empire"]["empty_colony_temperature_c"])

# Every owned colony, then generic planets (empty colony slots) until the empire has MAX_EMPIRE_PLANETS normal planets.
live_planet_states: list[PlanetState] = load_live_planet_states(LIVE_DB_PATH)
empire_planet_states: list[PlanetState] = build_padded_empire_planets(
    live_planet_states,
    max_planets=MAX_EMPIRE_PLANETS,
    empty_planet_size=EMPTY_COLONY_SIZE_FIELDS,
    empty_temp_c=EMPTY_COLONY_DEFAULT_TEMP_C,
)

# Account-wide inputs: your own research levels and the universe speed are the same on every planet.
ACCOUNT_SNAPSHOT: dict[str, Any] = load_account_snapshot(LIVE_DB_PATH)
colony_researches: dict[str, int] = ACCOUNT_SNAPSHOT["colony_researches"]
GAME_PARAM_CONTEXT: dict[str, Any] = {
    "speedup_factor": float(ACCOUNT_SNAPSHOT["universe_speedup_factor"]),
    "r1_mining_tech": colony_researches.get("r1", 0),
    "r2_energy_tech": colony_researches.get("r2", 0),
}

# One context per planet of the roster: the account-wide inputs and the planet's own values.
PLANET_PARAM_CONTEXTS: list[dict[str, Any]] = [
    planet_param_context(GAME_PARAM_CONTEXT, slot_index, planet, is_empty_slot=slot_index > len(live_planet_states))
    for slot_index, planet in enumerate(empire_planet_states, start=1)
]

print_section_data(
    "Section 3.1 data: account, planet roster and contexts",
    [
        (
            "empire settings",
            {
                "MAX_EMPIRE_PLANETS": MAX_EMPIRE_PLANETS,
                "EMPTY_COLONY_SIZE_FIELDS": EMPTY_COLONY_SIZE_FIELDS,
                "EMPTY_COLONY_DEFAULT_TEMP_C": EMPTY_COLONY_DEFAULT_TEMP_C,
            },
        ),
        ("ACCOUNT_SNAPSHOT", ACCOUNT_SNAPSHOT),
        ("colony_researches", colony_researches),
        ("GAME_PARAM_CONTEXT", GAME_PARAM_CONTEXT),
        ("live_planet_states", live_planet_states),
        ("empire_planet_states", empire_planet_states),
        ("PLANET_PARAM_CONTEXTS", PLANET_PARAM_CONTEXTS),
    ],
)

### 3.2 Maximum Research Levels in the Universe

The highest level of every research that anyone in the universe is known to have. Section 4.1 plans with these levels as its reference.

The bot keeps them in the `universe_research_max` table of its database. The notebook reads that ledger and combines it with the sources it can check itself:

- **Server records** (`server_record_level`) — the level that the game's statistics pages list for a research, once the bot has synchronized them into the ledger;
- **Spy reports** (`spied_max_level`) — the highest level found in the archived reports of `spy_reports`;
- **Your own research** (`own_level`) — the levels of the newest planet snapshot.

`universe_max_level` is the highest of these and of the maximum that the ledger stored earlier; the ledger never lowers a level. `max_source` names the source that holds the maximum (`server_record`, `spy_report`, `own_level`, `ledger` or `none`), and `holder` the player, when one is known. A level of 0 means that no source reported the research.

Only the bot fills the ledger; this notebook reads it. Until the bot has synchronized the server records, or spied a player whose report lists researches, every maximum is your own level.

- `universe_research_sources` — the raw inputs: the ledger rows, the highest spied level per research, and the number of spy reports that list researches;
- `universe_research_df` — one row per research: the level from each source, `universe_max_level`, `max_source`, `holder`, the distance from your own level (`gap_to_max`, `own_percent_of_max`), the last level of the catalog, and the time of the ledger's last update. A research that the ledger tracks but the catalog does not list comes last;
- `UNIVERSE_RESEARCH_MAX_LEVELS` — `{research key: universe_max_level}`.

In [ ]:
universe_research_sources: dict[str, Any] = load_universe_research_sources(LIVE_DB_PATH)
universe_research_df = build_universe_research_table(universe_research_sources, colony_researches)

# The highest known level of every research in the universe: the reference levels of the configuration optimizer.
UNIVERSE_RESEARCH_MAX_LEVELS: dict[str, int] = {
    str(key): int(level) for key, level in zip(universe_research_df["key"], universe_research_df["universe_max_level"])
}

print_section_data(
    "Section 3.2 data: maximum research levels in the universe",
    [
        ("universe_research_sources", universe_research_sources),
        ("universe_research_df", universe_research_df),
        ("UNIVERSE_RESEARCH_MAX_LEVELS", UNIVERSE_RESEARCH_MAX_LEVELS),
    ],
)

## 4. Best Empire Configuration for Imperial Starbase Production (Genetic Algorithm)

Which building levels, on which planets, build the most Imperial starbases per month? This section answers the question for the roster of Section 3.1: the colonies you own, plus the generic planets that stand for the colonies you can still found. Asteroids take no part.

**The model** describes a steady state: every building of the configuration stands, and each month's production is spent on the target ship.

- A planet's mines run at the share of the energy that the planet produces. Its Blast furnace smelts the planet's own Pig iron, 2 per Metal.
- Resources are pooled, so the empire pays for as many ships as its scarcest resource allows. Hauling resources to the shipyards, Merchant trades and raids are not modelled.
- Every planet has one shipyard queue. It builds the target ship only where the ship's building prerequisites are met (Weapons factory 13 for the Imperial starbase), at the speed of its Weapons factory and Microsystem accelerator.
- The number of ships per month is the smallest of four **limits**: the ships that the Metal, the Kryptonite and the Spice pay for, and the ships that the shipyards can finish.
- Every building level takes one field of its planet, and so does every warehouse level.

**Warehouses.** The model sets the four warehouses of every planet itself. Each one gets the lowest level that meets all of these rules:

1. It holds `WAREHOUSE_HOURS_OF_PRODUCTION` hours of the planet's own net production of its resource.
2. It holds the cost of the most expensive thing that the planet still has to pay for: a level of one of its buildings (a warehouse level included), a ship that it builds or, on the planet with the Research lab, a level of the planned research.
3. On the central planet, it also holds that many hours of the production of the whole empire, and it does not fall below `CENTRAL_PLANET_MIN_WAREHOUSE_LEVEL`.
4. It does not fall below its level in `MISC_BUILDING_LEVELS`, nor below the level already built.

Rule 2 is what limits the level of a mine. A warehouse holds twice as much with every level, and the last level of a mine costs twice as much about every 1.7 levels. A mine of level 50 therefore needs warehouses of level 18 to 21 for the resources it is paid in, and a mine of level 70 needs levels 30 to 33.

The cost of building the configuration does not enter the search. Section 4.4 reports it.

**Fixed and searched.** The meta parameters fix the research levels, the fleet and the miscellaneous buildings of every planet. The genetic algorithm decides the levels of the other buildings on every planet, and the warehouses follow from them. Each cell of this section prints its own data in full.

### 4.1 Planned Research and the Fixed Part of Every Planet

This cell turns the meta parameters of Section 1 into the inputs of the search, and prints them first. `check_optimizer_settings` stops with an error when they contradict the catalog or each other.

**Research.** `plan_research_levels` sets every research to `RESEARCH_PERCENT_OF_UNIVERSE_MAX` percent of its universe maximum, and raises it to your current level and to the level that the plan requires (`PLAN_REQUIRED_RESEARCH`: the prerequisites of the target ship, the fleet, and the buildings of the plan). In the model, Mining technology raises mine output and Energy technology raises energy output; the other researches only unlock ships and buildings.

**Planets.** `build_planet_plans` plans every normal planet of the roster:

- a miscellaneous building gets the highest of its meta-parameter level, the level already built, and the level that the planet's fleet and other miscellaneous buildings require;
- an optimized building may range from the level that those require (`min_level`) to its maximum (`max_level`). It stays at level 0, with the reason in `locked_by`, when the miscellaneous levels do not meet one of its own building prerequisites: the Microsystem accelerator needs Development center 12;
- a warehouse gets no level here, because its level depends on the design of the planet. The plan records its lowest level (`warehouse_min_levels`, with the reason in `warehouse_min_set_by`) and the most of each resource that the planet must be able to store, whatever its design (`capacity_needed`, with the cost that asks for it in `capacity_needed_for`). That is the cost of its fixed buildings and of the ships of its fleet. On the planet with the highest Research lab, it is also the cost of the planned research levels;
- the fields that the fixed buildings leave free are the budget of the optimized buildings and the warehouses (`free_fields`).

Tables and values:

- `WAREHOUSE_KEYS` — the warehouse of each resource;
- `planned_research_df` — per research: your level, the universe maximum, the percentage level, the required level, the planned level, and what set it (`set_by`: `current level`, `universe maximum`, `prerequisite` or `not needed`);
- `PLANNED_RESEARCH_LEVELS` — `{research key: planned level}`; `PLAN_ACCOUNT_CONTEXT` — the account context of Section 3.1 with these levels;
- `plan_planets_df` — per planet: whether it is the central planet and whether it pays for the research, fields, fields of the fixed buildings, free fields, the fields that the minimum levels of the optimized buildings and the lowest warehouses need, Solar cells counted, and the missing ships of the minimum fleet with their cost;
- `plan_building_levels_df` — per planet and building: fixed, optimized or warehouse, the level built today, and the level range of the plan;
- `PLANET_PLANS` — the same plan as one record per planet, the input of the model.

In [ ]:
# The warehouse of each resource, in `RESOURCE_KEYS` order.
WAREHOUSE_KEYS: tuple[str, ...] = tuple(
    str(RESOURCE_METADATA[resource]["warehouse_building_key"]) for resource in RESOURCE_KEYS
)


def same_planet_building_requirements(key: str, level: int = 0) -> dict[str, int]:
    """Building levels a planet needs before it can hold `key`, following building prerequisites only.

    `key` is a ship, a defense or a building; `level > 0` also demands that level of the building itself.
    Research prerequisites belong to the account, not to a planet: see `research_requirements`.
    """
    required: dict[str, int] = {}
    if level > 0 and TECH_GRAPH.nodes[key]["category"] == "buildings":
        required[key] = int(level)
    pending = [key]
    while pending:
        for source, source_level in TECH_GRAPH.prereqs_of[pending.pop()]:
            if TECH_GRAPH.nodes[source]["category"] == "buildings" and source_level > required.get(source, 0):
                required[source] = int(source_level)
                pending.append(source)
    return required


def research_requirements(keys: Iterable[str]) -> dict[str, int]:
    """Highest research level that any of `keys` (ships, defenses or buildings) needs, directly or indirectly."""
    required: dict[str, int] = {}
    for key in keys:
        for node, level in TECH_GRAPH.transitive_requirements(key).items():
            if TECH_GRAPH.nodes[node]["category"] == "research":
                required[node] = max(required.get(node, 0), int(level))
    return required


def catalog_top_level(key: str) -> int:
    """Last level of building `key` that the catalog lists."""
    info = game_data.BUILDING_BY_ID[int(key[1:])]
    return int(info.get("max_table_level") or len(info.get("levels") or {}))


def warehouse_level_for(amount: float) -> int:
    """Lowest warehouse level whose capacity holds `amount` units of a resource."""
    level = 0
    while warehouse_capacity(level) < amount:
        level += 1
    return level


def largest_costs(costs: Iterable[tuple[str, dict[str, Any]]]) -> tuple[dict[str, float], dict[str, str]]:
    """Per resource, the largest amount that any of the labelled `costs` asks for, and the label of that cost."""
    amounts = {resource: 0.0 for resource in RESOURCE_KEYS}
    labels = {resource: "" for resource in RESOURCE_KEYS}
    for label, cost in costs:
        for resource in RESOURCE_KEYS:
            amount = float(cost.get(resource, 0) or 0)
            if amount > amounts[resource]:
                amounts[resource], labels[resource] = amount, label
    return amounts, labels


def building_upgrade_costs(key: str, from_level: int, to_level: int) -> list[tuple[str, dict[str, Any]]]:
    """`(label, cost)` of every level of building `key` above `from_level`, up to `to_level`."""
    return [
        (f"{key} L{level}", game_data.building_cost(int(key[1:]), level))
        for level in range(from_level + 1, to_level + 1)
    ]


def research_upgrade_costs(
    own_levels: dict[str, int], planned_levels: dict[str, int]
) -> list[tuple[str, dict[str, Any]]]:
    """`(label, cost)` of every research level between the current levels and the planned ones."""
    return [
        (f"{key} L{level}", ((game_data.RESEARCH_BY_ID[key].get("levels") or {}).get(level) or {}).get("cost") or {})
        for key, planned_level in planned_levels.items()
        for level in range(max(0, int(own_levels.get(key, 0) or 0)) + 1, planned_level + 1)
    ]


def check_optimizer_settings(
    optimized_max_levels: dict[str, int],
    misc_levels: dict[str, int],
    misc_levels_by_slot: dict[int, dict[str, int]],
    min_fleet: dict[str, int],
    target_ship_key: str,
    warehouse_hours: float,
    central_min_warehouse_level: int,
) -> None:
    """Raise `ValueError` when the meta parameters contradict the catalog or each other."""
    building_keys = {f"b{building_id}" for building_id in game_data.BUILDING_BY_ID}
    optimized, fixed = set(optimized_max_levels), set(misc_levels)
    if (optimized | fixed) != building_keys or (optimized & fixed):
        raise ValueError(
            "OPTIMIZED_BUILDING_MAX_LEVELS and MISC_BUILDING_LEVELS must list every building exactly once: "
            f"missing {sorted(building_keys - optimized - fixed)}, "
            f"unknown {sorted((optimized | fixed) - building_keys)}, in both {sorted(optimized & fixed)}"
        )
    if optimized & set(WAREHOUSE_KEYS):
        raise ValueError(
            f"The warehouses {sorted(optimized & set(WAREHOUSE_KEYS))} belong in MISC_BUILDING_LEVELS: "
            "the plan sets their level to what each planet needs"
        )
    for slot, levels in misc_levels_by_slot.items():
        if not set(levels) <= fixed:
            raise ValueError(
                f"MISC_BUILDING_LEVELS_BY_SLOT[{slot}] sets {sorted(set(levels) - fixed)}, "
                "which MISC_BUILDING_LEVELS does not list"
            )
    if target_ship_key not in SHIP_SLUG_BY_KEY or not set(min_fleet) <= set(SHIP_SLUG_BY_KEY):
        raise ValueError(
            f"TARGET_SHIP_KEY and MIN_FLEET_PER_PLANET need ship keys: {sorted({target_ship_key, *min_fleet})}"
        )
    counts = [
        *optimized_max_levels.values(),
        *misc_levels.values(),
        *(level for levels in misc_levels_by_slot.values() for level in levels.values()),
        *min_fleet.values(),
        central_min_warehouse_level,
    ]
    if any(int(count) != count or count < 0 for count in counts):
        raise ValueError("Building levels and fleet sizes must be whole numbers of 0 or more")
    if not 0.0 <= warehouse_hours < math.inf:
        raise ValueError("WAREHOUSE_HOURS_OF_PRODUCTION must be a number of hours of 0 or more")
    for key in optimized_max_levels:
        linked = sorted((set(same_planet_building_requirements(key)) & optimized) - {key})
        if linked:
            raise ValueError(
                f"{key} needs {linked} on the same planet, and both are optimized: "
                "move one of them to MISC_BUILDING_LEVELS"
            )


def plan_research_levels(
    universe_max_levels: dict[str, int],
    own_levels: dict[str, int],
    required_levels: dict[str, int],
    percent_of_universe_max: float,
) -> pd.DataFrame:
    """Research levels of the plan: a share of the universe maximum, never below the current or required level."""
    rows: list[dict[str, Any]] = []
    for key, info in research_catalog():
        catalog_max = int(info.get("max_table_level") or len(info.get("levels") or {}))
        own_level = max(0, int(own_levels.get(key, 0) or 0))
        universe_max = int(universe_max_levels.get(key, 0))
        # Half a level rounds up: 50 % of level 5 is level 3.
        percent_level = min(catalog_max, int(math.floor(universe_max * float(percent_of_universe_max) / 100.0 + 0.5)))
        required_level = int(required_levels.get(key, 0))
        planned_level = max(own_level, percent_level, required_level)
        if planned_level == 0:
            set_by = "not needed"
        elif planned_level == own_level:
            set_by = "current level"
        elif planned_level == percent_level:
            set_by = "universe maximum"
        else:
            set_by = "prerequisite"
        rows.append(
            {
                "key": key,
                "name": str(info["name"]),
                "own_level": own_level,
                "universe_max_level": universe_max,
                "percent_level": percent_level,
                "required_level": required_level,
                "planned_level": planned_level,
                "set_by": set_by,
            }
        )
    return pd.DataFrame(rows)


def context_with_research(context: dict[str, Any], research_levels: dict[str, int]) -> dict[str, Any]:
    """Copy of an account context whose research entries (`r1_mining_tech`, ...) hold `research_levels`."""
    return {name: research_levels.get(name.partition("_")[0], value) for name, value in context.items()}


def build_planet_plan(
    context: dict[str, Any],
    planet: PlanetState,
    *,
    optimized_max_levels: dict[str, int],
    misc_levels: dict[str, int],
    min_fleet: dict[str, int],
    central_min_warehouse_level: int | None = None,
) -> dict[str, Any]:
    """Fixed part and search range of one planet.

    A miscellaneous building gets the highest of its meta-parameter level, the level already built, and the
    level that the planet's fleet and other miscellaneous buildings require. An optimized building ranges
    from the level those require up to its maximum; it stays at 0 (`locked_by`) when the miscellaneous
    levels do not meet one of its own building prerequisites.

    A warehouse is neither: the model sets its level for every design. The plan only gives its lowest level
    (`warehouse_min_levels`: the meta-parameter level, the level already built and, on the central planet,
    `central_min_warehouse_level`) and the most of each resource that the planet must be able to store to
    pay for its fixed buildings and its fleet (`capacity_needed`).
    """
    built = {key: max(0, int(planet.buildings.get(key, 0) or 0)) for key in (*misc_levels, *optimized_max_levels)}
    wanted = {key: max(int(level), built[key]) for key, level in misc_levels.items()}
    required: dict[str, int] = {}
    for needs in (
        *(same_planet_building_requirements(ship) for ship, count in min_fleet.items() if count > 0),
        *(same_planet_building_requirements(key, level) for key, level in wanted.items() if level > 0),
    ):
        for key, level in needs.items():
            required[key] = max(required.get(key, 0), level)
    fixed_levels = {
        key: max(level, required.get(key, 0)) for key, level in wanted.items() if key not in WAREHOUSE_KEYS
    }

    min_levels: dict[str, int] = {}
    max_levels: dict[str, int] = {}
    locked_by: dict[str, str] = {}
    for key, max_level in optimized_max_levels.items():
        unmet = [
            f"{source} L{level}"
            for source, level in same_planet_building_requirements(key).items()
            if fixed_levels.get(source, level) < level
        ]
        min_levels[key] = required.get(key, 0)
        max_levels[key] = max(min_levels[key], 0 if unmet else min(int(max_level), catalog_top_level(key)))
        if unmet:
            locked_by[key] = ", ".join(unmet)

    # The lowest level of each warehouse, and what sets it; between equal levels the first reason is named.
    warehouse_min_levels: dict[str, int] = {}
    warehouse_min_set_by: dict[str, str] = {}
    for key in WAREHOUSE_KEYS:
        level, reason = max(
            [
                (built[key], "level already built"),
                (int(central_min_warehouse_level or 0), "central planet minimum"),
                (int(misc_levels[key]), "MISC_BUILDING_LEVELS"),
            ],
            key=lambda candidate: candidate[0],
        )
        warehouse_min_levels[key], warehouse_min_set_by[key] = level, reason if level > 0 else "no minimum"

    stationed = {ship: max(0, int(planet.ships.get(ship, 0) or 0)) for ship in {*min_fleet, "s9"}}
    fleet_missing = {ship: max(0, int(count) - stationed[ship]) for ship, count in min_fleet.items()}
    capacity_needed, capacity_needed_for = largest_costs(
        [
            *(cost for key, level in fixed_levels.items() for cost in building_upgrade_costs(key, built[key], level)),
            *(
                (ship, game_data.SHIPS[SHIP_SLUG_BY_KEY[ship]].get("base_cost") or {})
                for ship, missing in fleet_missing.items()
                if missing > 0
            ),
        ]
    )
    fixed_fields = sum(fixed_levels.values())
    return {
        **planet_identity(context),
        "temperature_c": int(planet.temperature),
        "fields_max": int(planet.fields_max),
        "fixed_fields": fixed_fields,
        "free_fields": int(planet.fields_max) - fixed_fields,
        "built_levels": built,
        "fixed_levels": fixed_levels,
        "min_levels": min_levels,
        "max_levels": max_levels,
        "locked_by": locked_by,
        "warehouse_min_levels": warehouse_min_levels,
        "warehouse_min_set_by": warehouse_min_set_by,
        "capacity_needed": capacity_needed,
        "capacity_needed_for": capacity_needed_for,
        "is_central_planet": central_min_warehouse_level is not None,
        "pays_for_research": False,
        "solar_cells": max(int(min_fleet.get("s9", 0)), stationed["s9"]),
        "fleet_missing": fleet_missing,
        "fleet_cost_spice_eq": float(
            sum(
                count * game_data.spice_equivalent(full_cost(game_data.SHIPS[SHIP_SLUG_BY_KEY[ship]].get("base_cost")))
                for ship, count in fleet_missing.items()
            )
        ),
    }


def build_planet_plans(
    planets: list[PlanetState],
    contexts: list[dict[str, Any]],
    *,
    optimized_max_levels: dict[str, int],
    misc_levels: dict[str, int],
    misc_levels_by_slot: dict[int, dict[str, int]],
    min_fleet: dict[str, int],
    central_planet_slot: int,
    central_min_warehouse_level: int,
    research_costs: list[tuple[str, dict[str, Any]]],
) -> list[dict[str, Any]]:
    """Plan every normal planet of the roster; asteroids take no part in the optimization.

    The planet in roster slot `central_planet_slot` is the central planet. Research is paid for on the planet
    with the highest Research lab, the first of them between equals: its `capacity_needed` also covers the
    `research_costs`.
    """
    plans = [
        build_planet_plan(
            context,
            planet,
            optimized_max_levels=optimized_max_levels,
            misc_levels={**misc_levels, **misc_levels_by_slot.get(context["slot_index"], {})},
            min_fleet=min_fleet,
            central_min_warehouse_level=(
                central_min_warehouse_level if context["slot_index"] == central_planet_slot else None
            ),
        )
        for planet, context in zip(planets, contexts)
        if planet.planet_type == "normal"
    ]
    if not any(plan["is_central_planet"] for plan in plans):
        raise ValueError(
            f"CENTRAL_PLANET_SLOT {central_planet_slot} is not a normal planet of the roster: "
            f"choose one of the slots {[plan['slot_index'] for plan in plans]}"
        )
    lab_planet = max(plans, key=lambda plan: plan["fixed_levels"].get("b8", 0))
    if lab_planet["fixed_levels"].get("b8", 0) > 0:
        lab_planet["capacity_needed"], lab_planet["capacity_needed_for"] = largest_costs(
            [
                *(
                    (lab_planet["capacity_needed_for"][resource], {resource: lab_planet["capacity_needed"][resource]})
                    for resource in RESOURCE_KEYS
                ),
                *research_costs,
            ]
        )
        lab_planet["pays_for_research"] = True
    return plans


def planet_plan_level_rows(plan: dict[str, Any]) -> list[dict[str, Any]]:
    """Rows of `plan_building_levels_df` for one planet: the level range of every building."""
    rows: list[dict[str, Any]] = []
    for building_id in sorted(game_data.BUILDING_BY_ID):
        key = f"b{building_id}"
        if key in plan["min_levels"]:
            role, min_level, max_level = "optimized", plan["min_levels"][key], plan["max_levels"][key]
        elif key in plan["warehouse_min_levels"]:
            role, min_level, max_level = "warehouse", plan["warehouse_min_levels"][key], catalog_top_level(key)
        else:
            role, min_level, max_level = "fixed", plan["fixed_levels"][key], plan["fixed_levels"][key]
        rows.append(
            {
                "slot_index": plan["slot_index"],
                "coords": plan["coords"],
                "key": key,
                "name": str(game_data.BUILDING_BY_ID[building_id]["name"]),
                "role": role,
                "built_level": plan["built_levels"][key],
                "min_level": min_level,
                "max_level": max_level,
                "locked_by": plan["locked_by"].get(key, ""),
            }
        )
    return rows


check_optimizer_settings(
    OPTIMIZED_BUILDING_MAX_LEVELS,
    MISC_BUILDING_LEVELS,
    MISC_BUILDING_LEVELS_BY_SLOT,
    MIN_FLEET_PER_PLANET,
    TARGET_SHIP_KEY,
    WAREHOUSE_HOURS_OF_PRODUCTION,
    CENTRAL_PLANET_MIN_WAREHOUSE_LEVEL,
)

# Research: whatever the target ship, the fleet and the buildings of the plan need is the floor of the plan.
PLAN_REQUIRED_RESEARCH: dict[str, int] = research_requirements(
    [
        TARGET_SHIP_KEY,
        *(ship for ship, count in MIN_FLEET_PER_PLANET.items() if count > 0),
        *OPTIMIZED_BUILDING_MAX_LEVELS,
        *(key for key, level in MISC_BUILDING_LEVELS.items() if level > 0),
        *(key for levels in MISC_BUILDING_LEVELS_BY_SLOT.values() for key, level in levels.items() if level > 0),
    ]
)
planned_research_df = plan_research_levels(
    UNIVERSE_RESEARCH_MAX_LEVELS, colony_researches, PLAN_REQUIRED_RESEARCH, RESEARCH_PERCENT_OF_UNIVERSE_MAX
)
PLANNED_RESEARCH_LEVELS: dict[str, int] = {
    str(key): int(level) for key, level in zip(planned_research_df["key"], planned_research_df["planned_level"])
}
PLAN_ACCOUNT_CONTEXT: dict[str, Any] = context_with_research(GAME_PARAM_CONTEXT, PLANNED_RESEARCH_LEVELS)

# Planets: the fixed buildings, the level range of every optimized building, the lowest warehouses, and the
# fields left for the optimized buildings and the warehouses.
PLANET_PLANS: list[dict[str, Any]] = build_planet_plans(
    empire_planet_states,
    PLANET_PARAM_CONTEXTS,
    optimized_max_levels=OPTIMIZED_BUILDING_MAX_LEVELS,
    misc_levels=MISC_BUILDING_LEVELS,
    misc_levels_by_slot=MISC_BUILDING_LEVELS_BY_SLOT,
    min_fleet=MIN_FLEET_PER_PLANET,
    central_planet_slot=CENTRAL_PLANET_SLOT,
    central_min_warehouse_level=CENTRAL_PLANET_MIN_WAREHOUSE_LEVEL,
    research_costs=research_upgrade_costs(colony_researches, PLANNED_RESEARCH_LEVELS),
)
plan_planets_df = pd.DataFrame(
    [
        {
            **{
                column: plan[column]
                for column in (
                    "slot_index",
                    "planet_id",
                    "coords",
                    "planet_status",
                    "is_central_planet",
                    "pays_for_research",
                    "temperature_c",
                    "fields_max",
                    "fixed_fields",
                    "free_fields",
                )
            },
            "min_optimized_fields": sum(plan["min_levels"].values()),
            "min_warehouse_fields": sum(
                max(plan["warehouse_min_levels"][key], warehouse_level_for(plan["capacity_needed"][resource]))
                for resource, key in zip(RESOURCE_KEYS, WAREHOUSE_KEYS)
            ),
            "solar_cells": plan["solar_cells"],
            "fleet_units_missing": sum(plan["fleet_missing"].values()),
            "fleet_cost_spice_eq": round(plan["fleet_cost_spice_eq"], 2),
        }
        for plan in PLANET_PLANS
    ]
)
plan_building_levels_df = pd.DataFrame([row for plan in PLANET_PLANS for row in planet_plan_level_rows(plan)])

print_section_data(
    "Section 4.1 data: meta parameters, planned research and the fixed part of every planet",
    [
        ("TARGET_SHIP_KEY", TARGET_SHIP_KEY),
        ("PRODUCTION_MONTH_DAYS", PRODUCTION_MONTH_DAYS),
        ("RESEARCH_PERCENT_OF_UNIVERSE_MAX", RESEARCH_PERCENT_OF_UNIVERSE_MAX),
        ("MIN_FLEET_PER_PLANET", MIN_FLEET_PER_PLANET),
        ("MISC_BUILDING_LEVELS", MISC_BUILDING_LEVELS),
        ("MISC_BUILDING_LEVELS_BY_SLOT", MISC_BUILDING_LEVELS_BY_SLOT),
        ("WAREHOUSE_HOURS_OF_PRODUCTION", WAREHOUSE_HOURS_OF_PRODUCTION),
        ("CENTRAL_PLANET_SLOT", CENTRAL_PLANET_SLOT),
        ("CENTRAL_PLANET_MIN_WAREHOUSE_LEVEL", CENTRAL_PLANET_MIN_WAREHOUSE_LEVEL),
        ("OPTIMIZED_BUILDING_MAX_LEVELS", OPTIMIZED_BUILDING_MAX_LEVELS),
        (
            "search settings",
            {
                "CATALOGUE_GRID_SIZE": CATALOGUE_GRID_SIZE,
                "CATALOGUE_PRIORITY_STEPS": CATALOGUE_PRIORITY_STEPS,
                "HILL_CLIMB_STEP_SIZES": HILL_CLIMB_STEP_SIZES,
                "GA_POPULATION_SIZE": GA_POPULATION_SIZE,
                "GA_GENERATIONS": GA_GENERATIONS,
                "GA_TOURNAMENT_SIZE": GA_TOURNAMENT_SIZE,
                "GA_CROSSOVER_RATE": GA_CROSSOVER_RATE,
                "GA_MUTATION_RATE": GA_MUTATION_RATE,
                "GA_ELITE_COUNT": GA_ELITE_COUNT,
                "GA_RANDOM_SEED": GA_RANDOM_SEED,
            },
        ),
        ("WAREHOUSE_KEYS", WAREHOUSE_KEYS),
        ("PLAN_REQUIRED_RESEARCH", PLAN_REQUIRED_RESEARCH),
        ("planned_research_df", planned_research_df),
        ("PLANNED_RESEARCH_LEVELS", PLANNED_RESEARCH_LEVELS),
        ("PLAN_ACCOUNT_CONTEXT", PLAN_ACCOUNT_CONTEXT),
        ("plan_planets_df", plan_planets_df),
        ("plan_building_levels_df", plan_building_levels_df),
        ("PLANET_PLANS", PLANET_PLANS),
    ],
)

### 4.2 Production Model

`EmpireProductionModel` computes, for any building levels, what each planet adds to each limit. It is built once from `PLANET_PLANS` and then evaluates thousands of designs at a time with `numpy`. A **design** is one planet's levels of the optimized buildings; a **configuration** is one design per planet.

For a planet and a design, `simulate` works through the following steps, with the per-level numbers of Section 2.7 (`building_level_economy`) at the planned research levels:

1. Sum the hourly flows of the optimized and the fixed buildings: output per resource, Pig iron and Spice used as feedstock, energy produced and used. The Solar cells of the planet's fleet add their energy.
2. `power_ratio` = energy produced / energy used, at most 1. Every mine and the Blast furnace run at that share.
3. `smelting_ratio` = Pig iron mined / Pig iron the Blast furnace wants, at most 1. The furnace makes that share of its Metal.
4. The planet's net output per hour follows. Divided by the ship's cost, and counted over a month, it is the number of ships that the planet's Metal, Kryptonite or Spice pays for.
5. The shipyard finishes `month / (base time / (max(1, b9) * 2^b17 * speed))` ships, or none when the planet lacks a building that the ship requires.
6. The warehouses get the lowest levels that meet the rules of Section 4: the hours of the planet's net output, the most expensive level that the design still has to build (`upgrade_cost_table`), a target ship where the shipyard builds it, what the plan asks the planet to store, and the reserve of the central planet. A warehouse level has a cost of its own, so the levels are raised until the warehouses hold that as well.
7. The design fits the planet (`planet_fits`) when its levels and its warehouse levels together stay within the planet's free fields.

`planet_limits[n, planet, limit]` holds the numbers of steps 4 and 5, in ships per month. `ships_per_month` adds them up over the planets and takes the smallest sum, because resources are pooled. `is_feasible` checks the level ranges and the fields of a design.

The model stops with an error when a planet has no room for its minimum levels and their warehouses. `with_central_reserve` returns a copy of the model whose central planet keeps a given capacity free for the production of the empire; Section 4.3 uses it. `PRODUCTION_MODEL` itself reserves nothing yet.

Planets with identical inputs, such as the generic planets, are marked as twins (`twin_planets`), so that Section 4.3 searches designs for one of them only. The central planet has no twin.

In [ ]:
PRODUCTION_FLOW_KEYS: tuple[str, ...] = (
    *RESOURCE_KEYS,
    "pig_iron_used",
    "spice_used",
    "energy_produced",
    "energy_used",
)


def building_level_flows(building_key: str, level: int, context: dict[str, Any]) -> list[float]:
    """Hourly flows of one building level at full power on the context's planet, in `PRODUCTION_FLOW_KEYS` order."""
    economy = building_level_economy(int(building_key[1:]), level, context)
    output = {produced_resource(building_key): economy["production"]}
    return [
        *(float(output.get(resource, 0.0)) for resource in RESOURCE_KEYS),
        economy["pig_iron_used"],
        economy["spice_used"],
        max(0.0, economy["energy"]),
        max(0.0, -economy["energy"]),
    ]


def upgrade_cost_table(key: str, built_level: int, top_level: int) -> np.ndarray:
    """`table[level, resource]`: the most of each resource that a single level of building `key` costs on the way
    from `built_level` to `level`. It is 0 up to `built_level`, where nothing is left to build."""
    table = np.zeros((top_level + 1, len(RESOURCE_KEYS)), dtype=np.float64)
    for level in range(built_level + 1, top_level + 1):
        cost = full_cost(game_data.building_cost(int(key[1:]), level))
        table[level] = np.maximum(table[level - 1], [float(cost[resource]) for resource in RESOURCE_KEYS])
    return table


def upgrade_cost_tables(planet_plans: list[dict[str, Any]], keys: tuple[str, ...], top_level: int) -> np.ndarray:
    """`tables[planet, building, level, resource]`: the `upgrade_cost_table` of each of `keys` on every planet."""
    tables = {
        (key, built_level): upgrade_cost_table(key, built_level, top_level)
        for key in keys
        for built_level in {plan["built_levels"][key] for plan in planet_plans}
    }
    return np.array([[tables[key, plan["built_levels"][key]] for key in keys] for plan in planet_plans])


class EmpireProductionModel:
    """Steady-state output of an empire: what its planets add to each limit of the target ship's production.

    A design is one planet's levels of the optimized buildings, in `optimized_keys` order. The methods take
    many designs at once as `levels[n, planet, building]`; `planets` lists the planet of every column and
    defaults to all planets in order, which makes each `levels[n]` a configuration of the whole empire.
    The warehouses are not part of a design: the model gives every design the lowest warehouses it needs.
    """

    def __init__(
        self,
        planet_plans: list[dict[str, Any]],
        planet_contexts: list[dict[str, Any]],
        *,
        optimized_keys: tuple[str, ...],
        target_ship_key: str,
        month_days: float,
        warehouse_hours: float,
    ) -> None:
        self.optimized_keys = tuple(optimized_keys)
        self.target_ship_key = target_ship_key
        self.building_index = {key: index for index, key in enumerate(self.optimized_keys)}
        self.planet_names = [f"{plan['coords']} (slot {plan['slot_index']})" for plan in planet_plans]
        self.min_levels = np.array(
            [[plan["min_levels"][key] for key in self.optimized_keys] for plan in planet_plans], dtype=np.int64
        )
        self.max_levels = np.array(
            [[plan["max_levels"][key] for key in self.optimized_keys] for plan in planet_plans], dtype=np.int64
        )
        self.fields_max = np.array([plan["fields_max"] for plan in planet_plans], dtype=np.int64)
        # free_fields[planet]: the fields that the fixed buildings leave to the optimized ones and the warehouses.
        self.free_fields = np.array([plan["free_fields"] for plan in planet_plans], dtype=np.int64)
        self.fixed_levels = {
            key: np.array([plan["fixed_levels"][key] for plan in planet_plans], dtype=np.int64)
            for key in planet_plans[0]["fixed_levels"]
        }

        # flow_tables[flow, planet, building, level]: what each optimized building adds at each level.
        top_level = int(self.max_levels.max())
        self.flow_tables = np.array(
            [
                [
                    [building_level_flows(key, level, context) for level in range(top_level + 1)]
                    for key in self.optimized_keys
                ]
                for context in planet_contexts
            ],
            dtype=np.float64,
        ).transpose(3, 0, 1, 2)
        # fixed_flows[flow, planet]: the buildings at fixed levels and the Solar cells of the planet's fleet.
        self.fixed_flows = np.zeros((len(PRODUCTION_FLOW_KEYS), len(planet_plans)), dtype=np.float64)
        for planet, (plan, context) in enumerate(zip(planet_plans, planet_contexts)):
            for key, level in plan["fixed_levels"].items():
                self.fixed_flows[:, planet] += building_level_flows(key, level, context)
            self.fixed_flows[PRODUCTION_FLOW_KEYS.index("energy_produced"), planet] += (
                plan["solar_cells"]
                * solar_cell_energy_per_unit(context["planet_temperature_c"])
                * technology_multiplier("r2", context["r2_energy_tech"])
            )

        # Warehouses. warehouse_capacities[level] is what a warehouse of that level holds.
        self.warehouse_hours = float(warehouse_hours)
        self.top_warehouse_level = min(catalog_top_level(key) for key in WAREHOUSE_KEYS)
        self.warehouse_capacities = np.array(
            [warehouse_capacity(level) for level in range(self.top_warehouse_level + 1)], dtype=np.float64
        )
        self.min_warehouse_levels = np.array(
            [[plan["warehouse_min_levels"][key] for key in WAREHOUSE_KEYS] for plan in planet_plans], dtype=np.int64
        )
        # capacity_needed[planet, resource]: what the plan asks the planet to store, whatever its design.
        self.capacity_needed = np.array(
            [[plan["capacity_needed"][resource] for resource in RESOURCE_KEYS] for plan in planet_plans],
            dtype=np.float64,
        )
        # cost_tables[planet, building, level, resource] and warehouse_cost_tables[planet, warehouse, level, resource]:
        # the most that a single level still to build costs, for the optimized buildings and for the warehouses.
        self.cost_tables = upgrade_cost_tables(planet_plans, self.optimized_keys, top_level)
        self.warehouse_cost_tables = upgrade_cost_tables(planet_plans, WAREHOUSE_KEYS, self.top_warehouse_level)
        # The central planet also keeps central_reserve[resource] free for the production of the whole empire.
        self.central_planet = next(
            (planet for planet, plan in enumerate(planet_plans) if plan["is_central_planet"]), -1
        )
        self.central_reserve = np.zeros(len(RESOURCE_KEYS), dtype=np.float64)

        # The limits: one per resource the ship costs, then the shipyards.
        ship = game_data.SHIPS[SHIP_SLUG_BY_KEY[target_ship_key]]
        cost = full_cost(ship.get("base_cost"))
        self.cost_resources = tuple(resource for resource in RESOURCE_KEYS if float(cost[resource]) > 0.0)
        self.ship_cost = np.array([float(cost[resource]) for resource in self.cost_resources], dtype=np.float64)
        # ship_capacity[resource]: what a planet must store to pay for one target ship.
        self.ship_capacity = np.array([float(cost[resource]) for resource in RESOURCE_KEYS], dtype=np.float64)
        self.limit_names = (*self.cost_resources, "shipyards")
        self.target_requirements = same_planet_building_requirements(target_ship_key)
        self.hours_per_month = 24.0 * float(month_days)
        # shipyard_ships[weapons factory level, microsystem level]: target ships one shipyard finishes per month.
        factory_levels = max(top_level, int(self.fixed_levels.get("b9", np.zeros(1)).max())) + 1
        microsystem_levels = max(top_level, int(self.fixed_levels.get("b17", np.zeros(1)).max())) + 1
        speedup = float(planet_contexts[0]["speedup_factor"])
        seconds_per_month = 3600.0 * self.hours_per_month
        self.shipyard_ships = np.array(
            [
                [
                    seconds_per_month * shipyard_time_divisor(factory, microsystem, speedup)
                    / float(ship["base_build_time_s"])
                    for microsystem in range(microsystem_levels)
                ]
                for factory in range(factory_levels)
            ],
            dtype=np.float64,
        )

        # Planets with identical inputs, such as the generic planets, can share their results:
        # twin_planets[planet] is the first planet that equals it. The central planet has no twin.
        signatures = [
            (
                self.flow_tables[:, planet].tobytes(),
                self.fixed_flows[:, planet].tobytes(),
                self.min_levels[planet].tobytes(),
                self.max_levels[planet].tobytes(),
                int(self.free_fields[planet]),
                tuple(int(levels[planet]) for levels in self.fixed_levels.values()),
                self.min_warehouse_levels[planet].tobytes(),
                self.capacity_needed[planet].tobytes(),
                self.cost_tables[planet].tobytes(),
                self.warehouse_cost_tables[planet].tobytes(),
                planet == self.central_planet,
            )
            for planet in range(len(planet_plans))
        ]
        self.twin_planets = [signatures.index(signature) for signature in signatures]
        self.check_minimum_designs()

    def check_minimum_designs(self) -> None:
        """Raise `ValueError` when a planet has no room for its minimum levels and the warehouses they need."""
        outcome = self.simulate(self.min_levels[None])
        overfull = np.flatnonzero(~outcome["planet_fits"][0])
        if len(overfull) > 0:
            planet = int(overfull[0])
            if (outcome["planet_warehouse_levels"][0, planet] > self.top_warehouse_level).any():
                raise ValueError(
                    f"Planet {self.planet_names[planet]} would need a warehouse beyond level "
                    f"{self.top_warehouse_level}, the last level of the catalog: lower WAREHOUSE_HOURS_OF_PRODUCTION"
                )
            fixed_fields = int(self.fields_max[planet] - self.free_fields[planet])
            raise ValueError(
                f"Planet {self.planet_names[planet]} has {self.fields_max[planet]} fields, but its fixed buildings, "
                "the minimum levels of the optimized ones and their warehouses need "
                f"{fixed_fields + int(outcome['planet_fields_needed'][0, planet])}: lower MISC_BUILDING_LEVELS, "
                "CENTRAL_PLANET_MIN_WAREHOUSE_LEVEL or WAREHOUSE_HOURS_OF_PRODUCTION"
            )

    def with_central_reserve(self, capacity: np.ndarray) -> "EmpireProductionModel":
        """A copy of the model whose central planet keeps `capacity[resource]` free for the empire's production.

        The copy shares the tables of the model, which never change.
        """
        model = copy.copy(self)
        model.central_reserve = np.asarray(capacity, dtype=np.float64)
        model.check_minimum_designs()
        return model

    def planet_index(self, planets: Iterable[int] | None) -> np.ndarray:
        """The planet of every column of `levels[n, planet, building]`: `planets`, or all planets in order."""
        return np.arange(len(self.free_fields)) if planets is None else np.asarray(list(planets), dtype=np.int64)

    def building_levels(self, key: str, levels: np.ndarray, planets: np.ndarray) -> np.ndarray:
        """Level of building `key` for every design, whether the building is optimized or fixed."""
        if key in self.building_index:
            return levels[:, :, self.building_index[key]]
        return np.broadcast_to(self.fixed_levels[key][planets], levels.shape[:2])

    def warehouse_levels_for(self, capacity: np.ndarray) -> np.ndarray:
        """Lowest warehouse level that holds each amount of `capacity`; one above the top level when none does."""
        return np.searchsorted(self.warehouse_capacities, capacity)

    def simulate(self, levels: np.ndarray, planets: Iterable[int] | None = None) -> dict[str, np.ndarray]:
        """Evaluate designs `levels[n, planet, building]`; every result is indexed `[n, planet]` first.

        `planet_limits[n, planet, limit]` is what the planet adds to each limit, in target ships per month:
        the ships its net output of each cost resource pays for, and the ships its shipyard can finish.
        `planet_warehouse_levels[n, planet, resource]` are the warehouses that the design needs, and
        `planet_fits[n, planet]` says whether the design and those warehouses fit the planet's free fields.
        """
        planets = self.planet_index(planets)
        planet_axis = planets[None, :, None]
        building_axis = np.arange(levels.shape[2])[None, None, :]
        totals = (
            self.fixed_flows[:, None, planets]
            + self.flow_tables[:, planet_axis, building_axis, levels].sum(axis=3)
        )
        flows = dict(zip(PRODUCTION_FLOW_KEYS, totals))

        # A planet short of energy runs every mine at the share of the energy it has.
        power_ratio = np.ones_like(flows["energy_used"])
        np.divide(flows["energy_produced"], flows["energy_used"], out=power_ratio, where=flows["energy_used"] > 0.0)
        power_ratio = np.minimum(1.0, power_ratio)
        # The Blast furnace smelts the planet's own Pig iron, 2 per Metal, and idles for the part it lacks.
        pig_iron_mined = flows["pig_iron"] * power_ratio
        pig_iron_wanted = flows["pig_iron_used"] * power_ratio
        smelting_ratio = np.ones_like(pig_iron_wanted)
        np.divide(pig_iron_mined, pig_iron_wanted, out=smelting_ratio, where=pig_iron_wanted > 0.0)
        smelting_ratio = np.minimum(1.0, smelting_ratio)
        net = {
            "pig_iron": pig_iron_mined - pig_iron_wanted * smelting_ratio,
            "metal": flows["metal"] * power_ratio * smelting_ratio,
            "kryptonite": flows["kryptonite"] * power_ratio,
            "spice": flows["spice"] * power_ratio - flows["spice_used"],
        }
        net_per_hour = np.stack([net[resource] for resource in RESOURCE_KEYS], axis=2)

        # A shipyard builds the ship only where the ship's building prerequisites are met.
        can_build = np.ones(levels.shape[:2], dtype=bool)
        for key, required_level in self.target_requirements.items():
            can_build &= self.building_levels(key, levels, planets) >= required_level
        shipyard_ships = np.where(
            can_build,
            self.shipyard_ships[
                self.building_levels("b9", levels, planets), self.building_levels("b17", levels, planets)
            ],
            0.0,
        )
        planet_limits = np.stack(
            [
                *(
                    net[resource] * self.hours_per_month / cost
                    for resource, cost in zip(self.cost_resources, self.ship_cost)
                ),
                shipyard_ships,
            ],
            axis=2,
        )

        # Warehouses. Each one holds `warehouse_hours` of the planet's own net output, the most expensive level
        # that the design still has to build, one target ship where the shipyard builds it, what the plan asks
        # the planet to store and, on the central planet, the reserve for the empire.
        capacity = np.maximum(self.warehouse_hours * np.maximum(net_per_hour, 0.0), self.capacity_needed[planets])
        capacity = np.maximum(capacity, self.cost_tables[planet_axis, building_axis, levels].max(axis=2))
        capacity = np.maximum(capacity, np.where(can_build[:, :, None], self.ship_capacity, 0.0))
        capacity = np.maximum(
            capacity, np.where((planets == self.central_planet)[:, None], self.central_reserve, 0.0)
        )
        warehouses = np.maximum(self.warehouse_levels_for(capacity), self.min_warehouse_levels[planets])
        # A warehouse level costs resources too, and the warehouses must hold that cost: raise them until they do.
        warehouse_axis = np.arange(len(WAREHOUSE_KEYS))[None, None, :]
        while True:
            own_cost = self.warehouse_cost_tables[
                planet_axis, warehouse_axis, np.minimum(warehouses, self.top_warehouse_level)
            ].max(axis=2)
            raised = np.maximum(warehouses, self.warehouse_levels_for(own_cost))
            if (raised == warehouses).all():
                break
            warehouses = raised
        fields_needed = levels.sum(axis=2) + warehouses.sum(axis=2)
        return {
            "planet_limits": planet_limits,
            "planet_net_per_hour": net_per_hour,
            "planet_energy_produced": flows["energy_produced"],
            "planet_energy_used": flows["energy_used"],
            "planet_power_ratio": power_ratio,
            "planet_smelting_ratio": smelting_ratio,
            "planet_warehouse_levels": warehouses,
            "planet_fields_needed": fields_needed,
            "planet_fits": (fields_needed <= self.free_fields[planets])
            & (warehouses <= self.top_warehouse_level).all(axis=2),
        }

    def ships_per_month(self, levels: np.ndarray) -> np.ndarray:
        """Target ships per month of whole-empire configurations `levels[n, planet, building]`.

        Resources are pooled, so the empire builds what its scarcest limit allows: the sum over the planets of
        each limit, then the smallest of those sums.
        """
        return self.simulate(levels)["planet_limits"].sum(axis=1).min(axis=1)

    def in_range(self, levels: np.ndarray, planets: Iterable[int] | None = None) -> np.ndarray:
        """True per `levels[n]` when every level is inside the range of its building."""
        planets = self.planet_index(planets)
        return ((levels >= self.min_levels[planets]) & (levels <= self.max_levels[planets])).all(axis=(1, 2))

    def is_feasible(self, levels: np.ndarray, planets: Iterable[int] | None = None) -> np.ndarray:
        """True per `levels[n]` when every level is inside its range and every planet has the fields for its
        optimized buildings and the warehouses they need."""
        planets = self.planet_index(planets)
        inside = np.clip(levels, self.min_levels[planets], self.max_levels[planets])
        return self.in_range(levels, planets) & self.simulate(inside, planets)["planet_fits"].all(axis=1)


PRODUCTION_MODEL = EmpireProductionModel(
    PLANET_PLANS,
    [
        planet_param_context(
            PLAN_ACCOUNT_CONTEXT,
            plan["slot_index"],
            empire_planet_states[plan["slot_index"] - 1],
            is_empty_slot=plan["planet_status"] == "empty_colony_slot",
        )
        for plan in PLANET_PLANS
    ],
    optimized_keys=tuple(OPTIMIZED_BUILDING_MAX_LEVELS),
    target_ship_key=TARGET_SHIP_KEY,
    month_days=PRODUCTION_MONTH_DAYS,
    warehouse_hours=WAREHOUSE_HOURS_OF_PRODUCTION,
)

print_section_data(
    "Section 4.2 data: production model",
    [
        ("PRODUCTION_FLOW_KEYS", PRODUCTION_FLOW_KEYS),
        (
            "PRODUCTION_MODEL (EmpireProductionModel summary)",
            {
                "planets": len(PRODUCTION_MODEL.free_fields),
                "optimized_buildings": list(PRODUCTION_MODEL.optimized_keys),
                "limits": list(PRODUCTION_MODEL.limit_names),
                "ship_cost": dict(zip(PRODUCTION_MODEL.cost_resources, PRODUCTION_MODEL.ship_cost.tolist())),
                "ship_building_requirements": PRODUCTION_MODEL.target_requirements,
                "hours_per_month": PRODUCTION_MODEL.hours_per_month,
                "free_fields": PRODUCTION_MODEL.free_fields,
                "first_planet_with_the_same_inputs": PRODUCTION_MODEL.twin_planets,
                "warehouse_hours": PRODUCTION_MODEL.warehouse_hours,
                "central_planet": PRODUCTION_MODEL.planet_names[PRODUCTION_MODEL.central_planet],
            },
        ),
    ],
)

### 4.3 Design Catalogues, Genetic Algorithm and Polish

The search (`search_best_configuration`) has three steps.

**1. A catalogue of designs per planet** (`build_design_catalogues`). No single design is best for a planet: one planet may do most for the empire by making only Metal, another by making Kryptonite and Spice. The catalogue therefore holds one design for each mix of priorities between the limits, such as "only Metal", "Metal and Kryptonite in equal parts" or "a quarter each". With `CATALOGUE_PRIORITY_STEPS = 4` and four limits there are 35 mixes. For each mix, `planet_design_catalogue` takes the best of about `CATALOGUE_GRID_SIZE` coarse designs (`coarse_designs`) and improves it with a hill-climb (`improve_design`), which changes one, two or three buildings at a time by the `HILL_CLIMB_STEP_SIZES` and prefers fewer levels between equals. One more design balances all limits, for a planet that has to do everything itself. Only designs that fit the planet together with their warehouses are considered. Twin planets share a catalogue.

A shipyard counts only up to the ships that the empire could pay for if every planet made nothing but its scarcest resource (`resource_peaks`). A design that gives the shipyard any priority therefore gets the smallest shipyard that is large enough, and keeps the rest of its fields for mines.

**2. The genetic algorithm** (`run_genetic_algorithm`) decides which design each planet takes.

- A chromosome is one catalogue number per planet.
- Its fitness is the number of target ships per month of that configuration: the limits of the chosen designs, added up over the planets, and then the smallest sum.
- Each generation keeps the `GA_ELITE_COUNT` best chromosomes unchanged. Parents are picked by tournament: the best of `GA_TOURNAMENT_SIZE` random chromosomes. A child takes each planet's design from either parent (`GA_CROSSOVER_RATE` of the children; the others copy one parent), and then each of its planets switches to a random design of its catalogue with probability `GA_MUTATION_RATE`.
- `GA_RANDOM_SEED` makes the run repeatable.

**3. The polish** (`polish_both_ways`) hill-climbs the winner level by level, one planet at a time with the others held fixed, until a full round changes nothing (`polish_configuration`). It does this twice, once counting ships directly and once following `balanced_score` first, and keeps the better result. The polish can only add ships, and it removes levels that add none.

**Rounds for the central planet** (`search_in_rounds`). The central planet stores `WAREHOUSE_HOURS_OF_PRODUCTION` hours of the production of the whole empire, and that production is only known once a search has finished. The first round therefore reserves nothing for it. When the central planet of the result does not hold what the result produces (`central_reserve_needed`), the search runs again on a copy of the model that reserves that much on the central planet. The rounds end with the first result that the central planet can hold. `BEST_CONFIG_MODEL` is the model in which the central planet reserves exactly what that result produces; Sections 4.4 and 4.5 describe the result with it.

The result is a good configuration, not a proven optimum: a genetic algorithm gives no such guarantee. `ga_history_df` shows whether the run had settled; another `GA_RANDOM_SEED` or more `GA_GENERATIONS` are a cheap second opinion.

Data of this cell, all from the last round:

- `central_reserve_rounds_df` — one row per round: the ships per month, the capacity reserved on the central planet going in, the capacity that the result's production needs, and the capacity that the central planet of the result holds;
- `BEST_CONFIG_MODEL` — the production model whose central planet keeps room for the production of the result;
- `DESIGN_CATALOGUES` — per planet, the levels of every catalogue design; `design_catalogue_df` — the same designs with their warehouse levels, the fields they take, and what each adds to every limit, in ships per month;
- `ga_designs_arr` — the catalogue number that the algorithm chose for each planet; `ga_levels_arr` — the levels of that choice, per planet and optimized building;
- `ga_history_df` — best, mean and worst fitness of every generation;
- `best_levels_arr` — the levels after the polish: the configuration that Section 4.4 describes;
- `GA_SUMMARY` — the number of rounds, catalogue sizes, size of the search, and the ships per month before and after the polish.

In [ ]:
def balanced_score(limits: np.ndarray, sharpness: float = 8.0) -> np.ndarray:
    """Smooth stand-in for the smallest limit of each row of `limits[n, limit]`.

    It is the power mean of the row with exponent `-sharpness`. It never falls below the smallest limit and
    stays close to it, but unlike the smallest limit it also grows when any other limit grows. A search that
    follows it can trade a little of one limit for a lot of another, where the plain minimum would call
    every such step a loss.
    """
    floor = np.maximum(limits.min(axis=1, keepdims=True), 1e-9)
    return floor[:, 0] * np.mean((np.maximum(limits, 1e-9) / floor) ** -sharpness, axis=1) ** (-1.0 / sharpness)


def smallest_limit(limits: np.ndarray) -> np.ndarray:
    """The smallest limit of each row of `limits[n, limit]`: the target ships per month that the limits allow."""
    return limits.min(axis=1)


def planet_moves(buildings: int, step_sizes: tuple[int, ...]) -> np.ndarray:
    """Changes of one design that the hill-climb tries, as rows to add to its levels.

    For every step size: one building up or down; two buildings at once, each up or down; and two buildings
    up by the step while a third goes down by twice the step, which keeps the number of fields.
    """
    moves: set[tuple[int, ...]] = set()
    for size in step_sizes:
        for first in range(buildings):
            for first_step in (size, -size):
                moves.add(tuple(first_step if index == first else 0 for index in range(buildings)))
                for second in range(first + 1, buildings):
                    for second_step in (size, -size):
                        moves.add(
                            tuple(
                                first_step if index == first else second_step if index == second else 0
                                for index in range(buildings)
                            )
                        )
            for second in range(first + 1, buildings):
                for third in range(buildings):
                    if third not in (first, second):
                        moves.add(
                            tuple(
                                size if index in (first, second) else -2 * size if index == third else 0
                                for index in range(buildings)
                            )
                        )
    return np.array(sorted(moves), dtype=np.int64)


def improve_design(
    model: EmpireProductionModel,
    planet: int,
    design: np.ndarray,
    score: Any,
    moves: np.ndarray,
) -> np.ndarray:
    """Hill-climb one planet's design: keep taking the best neighbour until no neighbour is better.

    `score` maps the planet's limits `[n, limit]` to one number per row. A neighbour is better when it scores
    higher, or the same with fewer building levels, so the result carries no level that does nothing.
    Only neighbours that fit the planet, together with the warehouses they need, are considered.
    """
    current = np.asarray(design, dtype=np.int64)
    current_score = score(model.simulate(current[None, None, :], [planet])["planet_limits"][:, 0])[0]
    current_rank = (float(current_score), -int(current.sum()))
    while True:
        candidates = current + moves
        candidates = candidates[model.in_range(candidates[:, None, :], [planet])]
        if len(candidates) == 0:
            return current
        outcome = model.simulate(candidates[:, None, :], [planet])
        fits = outcome["planet_fits"][:, 0]
        if not fits.any():
            return current
        candidates, scores = candidates[fits], score(outcome["planet_limits"][fits, 0])
        level_totals = candidates.sum(axis=1)
        best = int(np.lexsort((-level_totals, scores))[-1])
        best_rank = (float(scores[best]), -int(level_totals[best]))
        if not best_rank > current_rank:  # also true for a score that is not a number, so the loop always ends
            return current
        current, current_rank = candidates[best], best_rank


def coarse_designs(model: EmpireProductionModel, planet: int, grid_size: int) -> tuple[np.ndarray, np.ndarray]:
    """A coarse grid of at most about `grid_size` designs of one planet: those that fit it, and their limits."""
    lower, upper = model.min_levels[planet], model.max_levels[planet]
    values_per_building = max(2, int(grid_size ** (1.0 / len(lower))))
    axes = [
        np.unique(np.linspace(low, high, values_per_building).round().astype(np.int64))
        for low, high in zip(lower, upper)
    ]
    grid = np.stack(np.meshgrid(*axes, indexing="ij"), axis=-1).reshape(-1, len(lower))
    outcome = model.simulate(grid[:, None, :], [planet])
    fits = outcome["planet_fits"][:, 0]
    return grid[fits], outcome["planet_limits"][fits, 0]


def resource_peaks(
    model: EmpireProductionModel,
    planet: int,
    grid: np.ndarray,
    grid_limits: np.ndarray,
    moves: np.ndarray,
) -> np.ndarray:
    """The most that one planet can add to each resource limit when it makes nothing else.

    For every resource limit, the best design of the coarse grid is hill-climbed with that limit as its score.
    """
    peaks: list[float] = []
    for column in range(len(model.cost_resources)):

        def this_limit(limits: np.ndarray, column: int = column) -> np.ndarray:
            return limits[:, column]

        design = improve_design(model, planet, grid[int(grid_limits[:, column].argmax())], this_limit, moves)
        peaks.append(float(model.simulate(design[None, None, :], [planet])["planet_limits"][0, 0, column]))
    return np.array(peaks)


def planet_design_catalogue(
    model: EmpireProductionModel,
    planet: int,
    grid: np.ndarray,
    grid_limits: np.ndarray,
    moves: np.ndarray,
    *,
    priority_steps: int,
    shipyard_ceiling: float,
) -> np.ndarray:
    """Candidate designs of one planet: for each mix of priorities between the limits, the design that serves it best.

    For every priority mix (each way to split `priority_steps` equal shares among the limits) the best design
    of the coarse grid is hill-climbed to a local optimum. One more design balances all limits, for a planet
    that has to do everything itself.

    A shipyard counts up to `shipyard_ceiling` ships and no further. Without that ceiling the shipyard limit,
    which doubles with each Microsystem accelerator level, would swallow the fields of every design that gives
    it any priority; with it, such a design gets the smallest shipyard that is large enough.
    """
    ceiling = np.full(grid_limits.shape[1], np.inf)
    ceiling[-1] = shipyard_ceiling
    # Each limit is measured against the best that a grid design reaches, so that the priorities compare like with like.
    scale = np.maximum(np.minimum(grid_limits, ceiling).max(axis=0), 1e-12)

    shares = np.stack(np.meshgrid(*[np.arange(priority_steps + 1)] * len(scale), indexing="ij"), axis=-1)
    shares = shares.reshape(-1, len(scale))
    designs: list[np.ndarray] = []
    for priorities in shares[shares.sum(axis=1) == priority_steps] / float(priority_steps):

        def priority_score(limits: np.ndarray, priorities: np.ndarray = priorities) -> np.ndarray:
            return (np.minimum(limits, ceiling) / scale) @ priorities

        start = grid[int(priority_score(grid_limits).argmax())]
        designs.append(improve_design(model, planet, start, priority_score, moves))
    start = grid[int(balanced_score(grid_limits).argmax())]
    designs.append(improve_design(model, planet, start, balanced_score, moves))
    return np.unique(np.array(designs), axis=0)


def build_design_catalogues(
    model: EmpireProductionModel,
    *,
    grid_size: int,
    priority_steps: int,
    step_sizes: tuple[int, ...],
) -> list[np.ndarray]:
    """One design catalogue per planet; a planet with a twin takes the twin's catalogue."""
    moves = planet_moves(len(model.optimized_keys), step_sizes)
    grids = {
        planet: coarse_designs(model, planet, grid_size)
        for planet, twin in enumerate(model.twin_planets)
        if twin == planet
    }
    # No empire builds more ships than its scarcest resource pays for when every planet makes nothing else:
    # that is the most a shipyard can be asked to finish.
    peaks = {planet: resource_peaks(model, planet, *grid, moves) for planet, grid in grids.items()}
    shipyard_ceiling = float(np.sum([peaks[twin] for twin in model.twin_planets], axis=0).min())
    catalogues = {
        planet: planet_design_catalogue(
            model, planet, *grid, moves, priority_steps=priority_steps, shipyard_ceiling=shipyard_ceiling
        )
        for planet, grid in grids.items()
    }
    return [catalogues[twin] for twin in model.twin_planets]


def catalogue_limits(model: EmpireProductionModel, catalogues: list[np.ndarray]) -> tuple[np.ndarray, np.ndarray]:
    """`design_limits[planet, design, limit]` of every catalogue design, and the number of designs per planet."""
    design_counts = np.array([len(catalogue) for catalogue in catalogues], dtype=np.int64)
    design_limits = np.zeros((len(catalogues), int(design_counts.max()), len(model.limit_names)), dtype=np.float64)
    for planet, catalogue in enumerate(catalogues):
        limits = model.simulate(catalogue[:, None, :], [planet])["planet_limits"][:, 0]
        design_limits[planet, : len(catalogue)] = limits
    return design_limits, design_counts


def design_catalogue_table(
    model: EmpireProductionModel,
    planet_plans: list[dict[str, Any]],
    catalogues: list[np.ndarray],
) -> pd.DataFrame:
    """One row per planet and catalogue design: its building levels, the fields that it takes together with its
    warehouses, and what it adds to each limit."""
    rows: list[dict[str, Any]] = []
    for planet, (plan, catalogue) in enumerate(zip(planet_plans, catalogues)):
        outcome = model.simulate(catalogue[:, None, :], [planet])
        for design, levels in enumerate(catalogue):
            rows.append(
                {
                    "slot_index": plan["slot_index"],
                    "coords": plan["coords"],
                    "design": design,
                    **dict(zip(model.optimized_keys, (int(level) for level in levels))),
                    "warehouse_levels": int(outcome["planet_warehouse_levels"][design, 0].sum()),
                    "fields": int(outcome["planet_fields_needed"][design, 0]),
                    **{
                        f"{limit}_ships_per_month": round(float(ships), 2)
                        for limit, ships in zip(model.limit_names, outcome["planet_limits"][design, 0])
                    },
                }
            )
    return pd.DataFrame(rows)


def run_genetic_algorithm(
    design_limits: np.ndarray,
    design_counts: np.ndarray,
    *,
    population_size: int,
    generations: int,
    tournament_size: int,
    crossover_rate: float,
    mutation_rate: float,
    elite_count: int,
    seed: int,
) -> tuple[np.ndarray, pd.DataFrame]:
    """Evolve the choice of one catalogue design per planet; return the best choice and the history.

    A chromosome holds one design number per planet. `design_limits[planet, design, limit]` is what that
    design adds to each limit, so the fitness of a chromosome, its target ships per month, is the smallest
    of the limit sums over its planets. Every generation keeps the `elite_count` best chromosomes, picks
    parents by tournament, takes each planet's design of a child from one of its two parents, and switches
    random planets of the children to a random design of their catalogue.
    """
    rng = np.random.default_rng(seed)
    planets = np.arange(len(design_counts))
    population = rng.integers(0, design_counts, size=(population_size, len(design_counts)))
    history: list[dict[str, Any]] = []
    for generation in range(generations + 1):
        fitness = design_limits[planets, population].sum(axis=1).min(axis=1)
        ranking = np.argsort(-fitness, kind="stable")
        history.append(
            {
                "generation": generation,
                "best_ships_per_month": float(fitness[ranking[0]]),
                "mean_ships_per_month": float(fitness.mean()),
                "worst_ships_per_month": float(fitness[ranking[-1]]),
            }
        )
        if generation == generations:
            break
        # Tournament selection: each parent is the best of `tournament_size` random chromosomes.
        contenders = rng.integers(0, population_size, size=(2 * population_size, tournament_size))
        winners = contenders[np.arange(2 * population_size), fitness[contenders].argmax(axis=1)]
        mothers, fathers = population[winners[:population_size]], population[winners[population_size:]]
        # Uniform crossover: a child takes each planet's design from its mother or its father.
        from_father = (rng.random(mothers.shape) < 0.5) & (rng.random((population_size, 1)) < crossover_rate)
        children = np.where(from_father, fathers, mothers)
        # Mutation: a planet switches to a random design of its catalogue.
        mutated = rng.random(children.shape) < mutation_rate
        children = np.where(mutated, rng.integers(0, design_counts, size=children.shape), children)
        population = np.concatenate([population[ranking[:elite_count]], children[: population_size - elite_count]])
    return population[ranking[0]].copy(), pd.DataFrame(history)


def polish_configuration(
    model: EmpireProductionModel,
    levels: np.ndarray,
    score: Any,
    step_sizes: tuple[int, ...],
) -> np.ndarray:
    """Hill-climb a whole configuration, one planet at a time, until a full round changes nothing.

    `score` maps the empire's limits `[n, limit]` to one number per row. Each planet's design is improved
    with `improve_design` while the other planets' contributions stay fixed.
    """
    moves = planet_moves(levels.shape[1], step_sizes)
    levels = np.array(levels, dtype=np.int64)
    planet_limits = model.simulate(levels[None])["planet_limits"][0]
    changed = True
    while changed:
        changed = False
        for planet in range(len(levels)):
            others = planet_limits.sum(axis=0) - planet_limits[planet]

            def empire_score(own: np.ndarray, others: np.ndarray = others) -> np.ndarray:
                return score(others + own)

            design = improve_design(model, planet, levels[planet], empire_score, moves)
            if (design != levels[planet]).any():
                levels[planet] = design
                planet_limits[planet] = model.simulate(design[None, None, :], [planet])["planet_limits"][0, 0]
                changed = True
    return levels


def polish_both_ways(model: EmpireProductionModel, levels: np.ndarray, step_sizes: tuple[int, ...]) -> np.ndarray:
    """Polish a configuration twice and keep the result with more target ships (fewer levels between equals).

    Once the polish counts ships directly. Once it first follows `balanced_score`, which can rearrange the
    planets more freely, and only then counts ships.
    """
    direct = polish_configuration(model, levels, smallest_limit, step_sizes)
    rebalanced = polish_configuration(
        model, polish_configuration(model, levels, balanced_score, step_sizes), smallest_limit, step_sizes
    )
    ships = model.ships_per_month(np.array([direct, rebalanced]))
    return rebalanced if (ships[1], -rebalanced.sum()) > (ships[0], -direct.sum()) else direct


def search_best_configuration(
    model: EmpireProductionModel,
    *,
    grid_size: int,
    priority_steps: int,
    step_sizes: tuple[int, ...],
    **ga_settings: Any,
) -> tuple[list[np.ndarray], np.ndarray, pd.DataFrame, np.ndarray, np.ndarray]:
    """One search on `model`: design catalogues, genetic algorithm, polish.

    Returns the catalogues, the catalogue design that the genetic algorithm chose for each planet, the history
    of its generations, the levels of that choice, and the levels after the polish. `ga_settings` are the
    keyword arguments of `run_genetic_algorithm`.
    """
    catalogues = build_design_catalogues(
        model, grid_size=grid_size, priority_steps=priority_steps, step_sizes=step_sizes
    )
    ga_designs, ga_history = run_genetic_algorithm(*catalogue_limits(model, catalogues), **ga_settings)
    ga_levels = np.array([catalogues[planet][design] for planet, design in enumerate(ga_designs)], dtype=np.int64)
    return catalogues, ga_designs, ga_history, ga_levels, polish_both_ways(model, ga_levels, step_sizes)


def central_reserve_needed(model: EmpireProductionModel, levels: np.ndarray) -> np.ndarray:
    """Capacity per resource that holds `warehouse_hours` of the net production of the whole empire."""
    net_per_hour = model.simulate(levels[None])["planet_net_per_hour"][0].sum(axis=0)
    return model.warehouse_hours * np.maximum(net_per_hour, 0.0)


def search_in_rounds(model: EmpireProductionModel, **search_settings: Any) -> tuple[Any, ...]:
    """Repeat the search until the warehouses of the central planet hold the production of the result.

    The central planet stores `warehouse_hours` of the production of the whole empire, and that production is
    only known once a search has finished. The first round therefore reserves nothing for it. Every further
    round reserves, on the central planet, room for the largest production that the rounds before came to.
    The rounds end with the first result whose central planet holds what the result itself produces; the
    reserve only grows from round to round, so they do end.

    Returns a copy of the model whose central planet reserves exactly what the last result produces, a table
    with one row per round, and the result of the last round as `search_best_configuration` returns it.
    `search_settings` are that function's keyword arguments.
    """
    reserve = np.zeros(len(RESOURCE_KEYS), dtype=np.float64)
    rounds: list[dict[str, Any]] = []
    while True:
        round_model = model.with_central_reserve(reserve)
        result = search_best_configuration(round_model, **search_settings)
        best_levels = result[-1]
        needed = central_reserve_needed(round_model, best_levels)
        warehouses = round_model.simulate(best_levels[None])["planet_warehouse_levels"][0]
        # A model without a central planet has nothing to hold.
        held = (
            round_model.warehouse_capacities[warehouses[round_model.central_planet]]
            if round_model.central_planet >= 0
            else np.full(len(RESOURCE_KEYS), np.inf)
        )
        holds_the_production = not (needed > held).any()
        rounds.append(
            {
                "round": len(rounds) + 1,
                "ships_per_month": round(float(round_model.ships_per_month(best_levels[None])[0]), 2),
                **{f"{resource}_reserved": float(amount) for resource, amount in zip(RESOURCE_KEYS, reserve)},
                **{f"{resource}_needed": round(float(amount), 1) for resource, amount in zip(RESOURCE_KEYS, needed)},
                **{f"{resource}_held": float(amount) for resource, amount in zip(RESOURCE_KEYS, held)},
                "holds_the_production": holds_the_production,
            }
        )
        if holds_the_production:
            return model.with_central_reserve(needed), pd.DataFrame(rounds), *result
        reserve = np.maximum(reserve, needed)


(
    BEST_CONFIG_MODEL,
    central_reserve_rounds_df,
    DESIGN_CATALOGUES,
    ga_designs_arr,
    ga_history_df,
    ga_levels_arr,
    best_levels_arr,
) = search_in_rounds(
    PRODUCTION_MODEL,
    grid_size=CATALOGUE_GRID_SIZE,
    priority_steps=CATALOGUE_PRIORITY_STEPS,
    step_sizes=HILL_CLIMB_STEP_SIZES,
    population_size=GA_POPULATION_SIZE,
    generations=GA_GENERATIONS,
    tournament_size=GA_TOURNAMENT_SIZE,
    crossover_rate=GA_CROSSOVER_RATE,
    mutation_rate=GA_MUTATION_RATE,
    elite_count=GA_ELITE_COUNT,
    seed=GA_RANDOM_SEED,
)
design_catalogue_df = design_catalogue_table(BEST_CONFIG_MODEL, PLANET_PLANS, DESIGN_CATALOGUES)

GA_SUMMARY: dict[str, Any] = {
    "search_rounds": int(len(central_reserve_rounds_df)),
    "designs_per_planet": [len(catalogue) for catalogue in DESIGN_CATALOGUES],
    "possible_choices_log10": round(float(sum(math.log10(len(catalogue)) for catalogue in DESIGN_CATALOGUES)), 2),
    "chromosomes_evaluated": GA_POPULATION_SIZE * (GA_GENERATIONS + 1) * int(len(central_reserve_rounds_df)),
    "first_generation_with_the_final_best": int(
        (ga_history_df["best_ships_per_month"] >= ga_history_df["best_ships_per_month"].iloc[-1]).idxmax()
    ),
    "ga_ships_per_month": round(float(BEST_CONFIG_MODEL.ships_per_month(ga_levels_arr[None])[0]), 2),
    "polished_ships_per_month": round(float(BEST_CONFIG_MODEL.ships_per_month(best_levels_arr[None])[0]), 2),
    "levels_changed_by_polish": int(np.abs(best_levels_arr - ga_levels_arr).sum()),
}

print_section_data(
    "Section 4.3 data: design catalogues, genetic algorithm and polish",
    [
        ("GA_SUMMARY", GA_SUMMARY),
        ("central_reserve_rounds_df", central_reserve_rounds_df),
        (
            "BEST_CONFIG_MODEL (capacity that the central planet keeps for the production of the empire)",
            dict(zip(RESOURCE_KEYS, BEST_CONFIG_MODEL.central_reserve.tolist())),
        ),
        ("ga_designs_arr", ga_designs_arr),
        ("ga_levels_arr", ga_levels_arr),
        ("best_levels_arr", best_levels_arr),
        ("ga_history_df", ga_history_df),
        ("design_catalogue_df", design_catalogue_df),
        ("DESIGN_CATALOGUES", DESIGN_CATALOGUES),
    ],
)

### 4.4 Best Configuration

The configuration that the search found, described by `build_best_configuration_tables` and `summarize_best_configuration`.

- `BEST_CONFIG_SUMMARY` — the result in numbers: target ships per month (`ships_per_month`, and the whole ships in `whole_ships_per_month`), the limit that holds the others back (`bottleneck`), the central planet, fields used and free, the fields that the warehouses take, the Pig iron that nothing uses, and the investment.
- `best_config_limits_df` — one row per limit: what one ship needs, what the empire has per month, the ships per month that it allows, and the share of it that the result uses. The bottleneck is used to 100 %.
- `best_config_buildings_df` — one row per planet: fields, and the level of every building: optimized, fixed or warehouse.
- `best_config_planets_df` — one row per planet: energy produced and used, `power_ratio` and `smelting_ratio` (1 means that nothing is held back), net output per hour of each resource, the ships that its shipyard could finish and the ships it builds, the fields of its warehouses, and the cost of getting there.
- `best_config_warehouses_df` — one row per planet and resource, from `warehouse_requirement_rows`: the level of the warehouse, its capacity, and the level that each rule of Section 4 asks for: the largest cost (`level_for_largest_cost`, with the level or ship that costs it in `largest_cost_of`), the hours of the planet's own production, the hours of the empire's production on the central planet, and the minimum level. `set_by` names the rule that asks for the most.
- `BEST_CONFIG_TABLES` — registry of the four tables.

**Investment.** The search maximizes monthly output and ignores what the buildings cost, so the result is limited only by the fields, the energy, the warehouses and the maximum levels of Section 1. The summary puts a price on it, in Spice-equivalent at Merchant parity:

- `buildings_cost_spice_eq` — every building level that is still missing on your colonies and on the generic planets (`levels_to_build`), warehouses included;
- `fleet_cost_spice_eq` — the ships missing from the minimum fleets;
- `research_cost_spice_eq` — the research levels between your current ones and the planned ones;
- `investment_in_months_of_output` — the sum of the three, divided by the value of the ships that the finished configuration builds in a month.

When that last number is large, lower `OPTIMIZED_BUILDING_MAX_LEVELS` to see the best configuration within reach.

In [ ]:
def building_upgrade_cost_spice_eq(key: str, from_level: int, to_level: int) -> float:
    """Spice-equivalent cost of raising building `key` from `from_level` to `to_level`; 0 when it is not higher."""
    return float(
        sum(
            game_data.spice_equivalent(game_data.building_cost(int(key[1:]), level))
            for level in range(from_level + 1, to_level + 1)
        )
    )


def research_upgrade_cost_spice_eq(key: str, from_level: int, to_level: int) -> float:
    """Spice-equivalent cost of raising research `key` from `from_level` to `to_level`; 0 when it is not higher."""
    levels = game_data.RESEARCH_BY_ID[key].get("levels") or {}
    return float(
        sum(
            game_data.spice_equivalent(full_cost((levels.get(level) or {}).get("cost")))
            for level in range(from_level + 1, to_level + 1)
        )
    )


def warehouse_requirement_rows(
    plan: dict[str, Any],
    planned_levels: dict[str, int],
    net_per_hour: np.ndarray,
    *,
    warehouse_hours: float,
    ship_built: str | None,
    central_reserve: np.ndarray,
) -> list[dict[str, Any]]:
    """Why each warehouse of a planet has its level: one row per resource, with the level that every rule asks for.

    `planned_levels` holds the planet's level of every building and `net_per_hour` its net output of each
    resource. `ship_built` is the key of the target ship when the planet's shipyard builds it, and
    `central_reserve` the capacity per resource that the planet keeps for the empire: zeros, unless it is the
    central planet.
    """
    largest_cost, largest_cost_of = largest_costs(
        [
            *(
                (plan["capacity_needed_for"][resource], {resource: plan["capacity_needed"][resource]})
                for resource in RESOURCE_KEYS
            ),
            *(
                cost
                for key in (*plan["min_levels"], *WAREHOUSE_KEYS)
                for cost in building_upgrade_costs(key, plan["built_levels"][key], planned_levels[key])
            ),
            *(
                (ship, game_data.SHIPS[SHIP_SLUG_BY_KEY[ship]].get("base_cost") or {})
                for ship in ([ship_built] if ship_built else [])
            ),
        ]
    )
    rows: list[dict[str, Any]] = []
    for index, (resource, key) in enumerate(zip(RESOURCE_KEYS, WAREHOUSE_KEYS)):
        # The level that each rule asks for, with its reason; between equal levels the first reason is named.
        asked = [
            (warehouse_level_for(largest_cost[resource]), f"cost of {largest_cost_of[resource]}"),
            (warehouse_level_for(warehouse_hours * max(0.0, float(net_per_hour[index]))), "own production"),
            (warehouse_level_for(float(central_reserve[index])), "empire production"),
            (plan["warehouse_min_levels"][key], plan["warehouse_min_set_by"][key]),
        ]
        level, set_by = max(asked, key=lambda rule: rule[0])
        rows.append(
            {
                "slot_index": plan["slot_index"],
                "coords": plan["coords"],
                "resource": resource,
                "key": key,
                "built_level": plan["built_levels"][key],
                "level": planned_levels[key],
                "capacity": float(warehouse_capacity(planned_levels[key])),
                "largest_cost": largest_cost[resource],
                "largest_cost_of": largest_cost_of[resource],
                "level_for_largest_cost": asked[0][0],
                "level_for_own_production": asked[1][0],
                "level_for_empire_production": asked[2][0],
                "minimum_level": asked[3][0],
                "set_by": set_by if level > 0 else "nothing to store",
            }
        )
    return rows


def build_best_configuration_tables(
    model: EmpireProductionModel,
    planet_plans: list[dict[str, Any]],
    levels: np.ndarray,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Describe one configuration: building levels per planet, each planet's output, the empire's limits, and
    the reason for the level of every warehouse."""
    outcome = {name: values[0] for name, values in model.simulate(levels[None]).items()}
    limit_totals = outcome["planet_limits"].sum(axis=0)
    ships = float(limit_totals.min())
    shipyard_column = model.limit_names.index("shipyards")
    building_rows: list[dict[str, Any]] = []
    planet_rows: list[dict[str, Any]] = []
    warehouse_rows: list[dict[str, Any]] = []
    for index, plan in enumerate(planet_plans):
        identity = {column: plan[column] for column in ("slot_index", "planet_id", "coords", "planet_status")}
        warehouses = dict(zip(WAREHOUSE_KEYS, (int(level) for level in outcome["planet_warehouse_levels"][index])))
        planned = {
            **plan["fixed_levels"],
            **warehouses,
            **dict(zip(model.optimized_keys, (int(level) for level in levels[index]))),
        }
        fields_used = sum(planned.values())
        building_rows.append(
            {
                **identity,
                "fields_max": plan["fields_max"],
                "fields_used": fields_used,
                "fields_free": plan["fields_max"] - fields_used,
                **{f"b{building_id}": planned[f"b{building_id}"] for building_id in sorted(game_data.BUILDING_BY_ID)},
            }
        )
        shipyard_ships = float(outcome["planet_limits"][index, shipyard_column])
        shipyard_total = float(limit_totals[shipyard_column])
        planet_rows.append(
            {
                **identity,
                "energy_produced": round(float(outcome["planet_energy_produced"][index]), 1),
                "energy_used": round(float(outcome["planet_energy_used"][index]), 1),
                "power_ratio": round(float(outcome["planet_power_ratio"][index]), 4),
                "smelting_ratio": round(float(outcome["planet_smelting_ratio"][index]), 4),
                **{
                    f"{resource}_per_hour": round(float(outcome["planet_net_per_hour"][index, column]), 1)
                    for column, resource in enumerate(RESOURCE_KEYS)
                },
                "shipyard_ships_per_month": round(shipyard_ships, 2),
                # The shipyards share the ships that the resources pay for in proportion to their capacity.
                "ships_built_per_month": (
                    round(shipyard_ships * ships / shipyard_total, 2) if shipyard_total > 0.0 else 0.0
                ),
                "warehouse_fields": sum(warehouses.values()),
                "levels_to_build": sum(max(0, level - plan["built_levels"][key]) for key, level in planned.items()),
                "upgrade_cost_spice_eq": round(
                    sum(
                        building_upgrade_cost_spice_eq(key, plan["built_levels"][key], level)
                        for key, level in planned.items()
                    ),
                    2,
                ),
                "fleet_cost_spice_eq": round(plan["fleet_cost_spice_eq"], 2),
            }
        )
        warehouse_rows.extend(
            warehouse_requirement_rows(
                plan,
                planned,
                outcome["planet_net_per_hour"][index],
                warehouse_hours=model.warehouse_hours,
                ship_built=model.target_ship_key if shipyard_ships > 0.0 else None,
                central_reserve=(
                    model.central_reserve if index == model.central_planet else np.zeros(len(RESOURCE_KEYS))
                ),
            )
        )

    # Per ship: the units of a resource that one ship costs; a shipyard finishes whole ships, so it "costs" 1.
    limits = pd.DataFrame(
        {
            "limit": list(model.limit_names),
            "needed_per_ship": [*model.ship_cost.tolist(), 1.0],
            "ships_per_month": np.round(limit_totals, 2),
        }
    )
    limits["available_per_month"] = np.round(limit_totals * limits["needed_per_ship"].to_numpy(), 1)
    limits["used_percent"] = [round(100.0 * ships / total, 2) if total > 0.0 else 0.0 for total in limit_totals]
    limits["is_bottleneck"] = limit_totals == limit_totals.min()
    return pd.DataFrame(building_rows), pd.DataFrame(planet_rows), limits, pd.DataFrame(warehouse_rows)


def summarize_best_configuration(
    model: EmpireProductionModel,
    levels: np.ndarray,
    buildings: pd.DataFrame,
    planets: pd.DataFrame,
    limits: pd.DataFrame,
    own_research: dict[str, int],
    planned_research: dict[str, int],
) -> dict[str, Any]:
    """Headline numbers of a configuration: ships per month, the bottleneck, and what building it would cost."""
    ship = game_data.SHIPS[SHIP_SLUG_BY_KEY[model.target_ship_key]]
    ship_cost_spice_eq = float(game_data.spice_equivalent(full_cost(ship.get("base_cost"))))
    ships = float(model.ships_per_month(levels[None])[0])
    costs = {
        "buildings_cost_spice_eq": float(planets["upgrade_cost_spice_eq"].sum()),
        "fleet_cost_spice_eq": float(planets["fleet_cost_spice_eq"].sum()),
        "research_cost_spice_eq": float(
            sum(
                research_upgrade_cost_spice_eq(key, max(0, int(own_research.get(key, 0) or 0)), level)
                for key, level in planned_research.items()
            )
        ),
    }
    investment = sum(costs.values())
    monthly_output = ships * ship_cost_spice_eq
    return {
        "target_ship": str(ship["name"]),
        "ships_per_month": round(ships, 2),
        "whole_ships_per_month": int(ships),
        "bottleneck": ", ".join(limits.loc[limits["is_bottleneck"], "limit"]),
        "hours_per_month": model.hours_per_month,
        "planets": int(len(buildings)),
        "central_planet": model.planet_names[model.central_planet] if model.central_planet >= 0 else "",
        "fields_used": int(buildings["fields_used"].sum()),
        "fields_free": int(buildings["fields_free"].sum()),
        "warehouse_fields": int(planets["warehouse_fields"].sum()),
        "pig_iron_left_over_per_month": round(float(planets["pig_iron_per_hour"].sum()) * model.hours_per_month, 1),
        "levels_to_build": int(planets["levels_to_build"].sum()),
        **{name: round(cost, 2) for name, cost in costs.items()},
        "investment_spice_eq": round(investment, 2),
        "ship_cost_spice_eq": round(ship_cost_spice_eq, 2),
        "monthly_output_spice_eq": round(monthly_output, 2),
        "investment_in_months_of_output": round(investment / monthly_output, 2) if monthly_output > 0.0 else None,
    }


(
    best_config_buildings_df,
    best_config_planets_df,
    best_config_limits_df,
    best_config_warehouses_df,
) = build_best_configuration_tables(BEST_CONFIG_MODEL, PLANET_PLANS, best_levels_arr)
BEST_CONFIG_SUMMARY: dict[str, Any] = summarize_best_configuration(
    BEST_CONFIG_MODEL,
    best_levels_arr,
    best_config_buildings_df,
    best_config_planets_df,
    best_config_limits_df,
    colony_researches,
    PLANNED_RESEARCH_LEVELS,
)
BEST_CONFIG_TABLES: dict[str, pd.DataFrame] = {
    "best_config_limits": best_config_limits_df,
    "best_config_buildings": best_config_buildings_df,
    "best_config_planets": best_config_planets_df,
    "best_config_warehouses": best_config_warehouses_df,
}

print_section_data(
    f"Section 4.4 data: best configuration, {BEST_CONFIG_SUMMARY['ships_per_month']:,.2f} "
    f"{BEST_CONFIG_SUMMARY['target_ship']}s per month (bottleneck: {BEST_CONFIG_SUMMARY['bottleneck']})",
    [
        ("BEST_CONFIG_SUMMARY", BEST_CONFIG_SUMMARY),
        ("best_config_limits_df", best_config_limits_df),
        ("best_config_buildings_df", best_config_buildings_df),
        ("best_config_planets_df", best_config_planets_df),
        ("best_config_warehouses_df", best_config_warehouses_df),
        ("BEST_CONFIG_TABLES", BEST_CONFIG_TABLES),
    ],
)

### 4.5 Configuration of Each Planet

What to build on each planet to reach the result of Section 4.4. `PLANET_CONFIGURATIONS` holds one table per planet, built by `planet_configuration_table` and printed one after the other. The label of a table names the roster slot, the coordinates, the status of the planet and the fields that the configuration uses. A table lists every building that the planet has today or gets in the configuration:

- `built_level` — the level that stands today (0 everywhere on a generic planet);
- `target_level` — the level in the best configuration;
- `levels_to_build`, `upgrade_cost_spice_eq` — the levels still missing, and their cost in Spice-equivalent at Merchant parity;
- `last_level_build_time_s`, `last_level_build_time_text` — how long the last of the missing levels takes to build (`last_level_build_time`), in seconds and as text. Both are 0 for a building that has no level left to build;
- `role` — `optimized` when the genetic algorithm chose the level, `fixed` when the meta parameters set it, and `warehouse` when the model set it to what the planet needs (`best_config_warehouses_df` of Section 4.4 gives the reason).

**Build time.** The time of a level is its base time from the catalog, divided by `(Development center + 1) * 2^Microsystem accelerator * universe speed` (`building_time_divisor` of Section 2.3). The last level is timed with the Development center and the Microsystem accelerator that the planet has in the configuration, so it is the time at the end of the build-up; earlier in the build-up the same level would take longer. While one of these two buildings is upgraded itself, it still works at its level before. The catalog lost the exponent of some base times at very high levels; `build_monotonic_base_times` of Section 2.3 rebuilds those from the level before and the growth of the cost.

In [ ]:
def last_level_build_time(key: str, planned_levels: dict[str, int], speedup_factor: float) -> float:
    """Seconds that the last level of building `key` takes on a planet whose buildings reach `planned_levels`.

    The level is built with the Development center and the Microsystem accelerator of the plan. While one of
    these two is upgraded itself, it still works at the level before.
    """
    level = planned_levels[key]
    if level <= 0:
        return 0.0
    base_times = build_monotonic_base_times(game_data.BUILDING_BY_ID[int(key[1:])].get("levels") or {}, level)
    dev_center_level = planned_levels["b7"] - (1 if key == "b7" else 0)
    microsystem_level = planned_levels["b17"] - (1 if key == "b17" else 0)
    return base_times[level] / building_time_divisor(dev_center_level, microsystem_level, speedup_factor)


def planet_configuration_table(
    plan: dict[str, Any],
    planned_levels: dict[str, int],
    speedup_factor: float,
) -> pd.DataFrame:
    """One planet's configuration: every building it has or gets, the level to reach, and what is still missing."""
    rows: list[dict[str, Any]] = []
    for building_id in sorted(game_data.BUILDING_BY_ID):
        key = f"b{building_id}"
        built_level, target_level = plan["built_levels"][key], planned_levels[key]
        if built_level == 0 and target_level == 0:
            continue
        if key in plan["min_levels"]:
            role = "optimized"
        else:
            role = "warehouse" if key in WAREHOUSE_KEYS else "fixed"
        # A building that already stands at its target level has no level left to build, and so no build time.
        build_time = last_level_build_time(key, planned_levels, speedup_factor) if target_level > built_level else 0.0
        rows.append(
            {
                "key": key,
                "building": str(game_data.BUILDING_BY_ID[building_id]["name"]),
                "role": role,
                "built_level": built_level,
                "target_level": target_level,
                "levels_to_build": max(0, target_level - built_level),
                "upgrade_cost_spice_eq": round(building_upgrade_cost_spice_eq(key, built_level, target_level), 2),
                "last_level_build_time_s": round(build_time, 2),
                "last_level_build_time_text": fmt_duration(build_time),
            }
        )
    return pd.DataFrame(rows)


# One table per planet, in roster order. The label says which planet it is and how many fields it uses.
PLANET_CONFIGURATIONS: dict[str, pd.DataFrame] = {
    f"slot {plan['slot_index']}, {plan['coords']} ({plan['planet_status']}, "
    f"{int(levels['fields_used'])} of {plan['fields_max']} fields)": planet_configuration_table(
        plan,
        {f"b{building_id}": int(levels[f"b{building_id}"]) for building_id in sorted(game_data.BUILDING_BY_ID)},
        float(PLAN_ACCOUNT_CONTEXT["speedup_factor"]),
    )
    for plan, (_, levels) in zip(PLANET_PLANS, best_config_buildings_df.iterrows())
}

print_section_data(
    f"Section 4.5 data: configuration of each planet for {BEST_CONFIG_SUMMARY['ships_per_month']:,.2f} "
    f"{BEST_CONFIG_SUMMARY['target_ship']}s per month",
    [(f"PLANET_CONFIGURATIONS: {label}", table) for label, table in PLANET_CONFIGURATIONS.items()],
)

## 5. Save the Result as JSON

The last cell writes the result of the notebook into `METADATA_DIR`, next to the static metadata, as `config_optimizer_result.json`. It is the only file that this notebook writes. Every run replaces it, so the file always describes the latest run; `generated_at` says when that was.

`CONFIG_RESULT` is the saved document:

| Key | Content |
|---|---|
| `description`, `schema_version`, `generated_at` | what the file is, the version of its layout, and the local date and time of the run |
| `meta_parameters` | every meta parameter of Section 1, as it was set for the run |
| `inputs` | the universe speed, the number of owned planets and of planets in the roster, your research levels, and the universe maximum of every research |
| `planned_research_levels` | the research levels of the plan |
| `summary` | `BEST_CONFIG_SUMMARY`: the ships per month, the bottleneck, the fields and the investment |
| `search` | `GA_SUMMARY`: the size of the search and what the polish added |
| `limits` | the rows of `best_config_limits_df` |
| `planets` | one record per planet, in roster order |

A planet record holds what identifies the planet (`slot_index`, `planet_id`, `coords`, `planet_status`), whether it is the central planet and whether it pays for the research, its temperature, its fields, and three parts:

- `production` — its row of `best_config_planets_df`: energy, output per hour, ships, and the cost of getting there;
- `warehouses` — its rows of `best_config_warehouses_df`: the level of each warehouse and the rule that set it;
- `buildings` — its table of Section 4.5: every building with the level built today, the target level, the levels still to build, their cost, and the build time of the last level.

`json_ready` turns the values into plain JSON data: numpy numbers become Python numbers, tuples become lists, the roster slots that key `MISC_BUILDING_LEVELS_BY_SLOT` become text, and a number that is not finite becomes `null`. `table_records` turns a table into one object per row, and `planet_result_records` builds the planet records.

The file holds the coordinates and the building levels of your planets.

In [ ]:
def json_ready(value: Any) -> Any:
    """`value` as plain JSON data: objects with text keys, lists, text, numbers, booleans and null.

    numpy values become Python values and tuples become lists. A number that is not finite becomes null,
    because JSON has no way to write it.
    """
    if isinstance(value, dict):
        return {str(key): json_ready(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_ready(item) for item in value]
    if isinstance(value, np.ndarray):
        return json_ready(value.tolist())
    if isinstance(value, np.generic):
        return json_ready(value.item())
    if isinstance(value, float) and not math.isfinite(value):
        return None
    if value is None or isinstance(value, (str, int, float)):
        return value
    raise TypeError(f"Cannot save a value of type {type(value).__name__} as JSON: {value!r}")


def table_records(table: pd.DataFrame) -> list[dict[str, Any]]:
    """The rows of a table as JSON data: one object per row, keyed by column."""
    return json_ready(table.to_dict(orient="records"))


def planet_result_records(
    planet_plans: list[dict[str, Any]],
    buildings: pd.DataFrame,
    planets: pd.DataFrame,
    warehouses: pd.DataFrame,
    configurations: dict[str, pd.DataFrame],
) -> list[dict[str, Any]]:
    """One record per planet, in roster order: what identifies it, its fields, production, warehouses and buildings.

    `buildings`, `planets` and `warehouses` are the tables of Section 4.4, and `configurations` the per-planet
    tables of Section 4.5.
    """
    identity = ("slot_index", "planet_id", "coords", "planet_status")
    records: list[dict[str, Any]] = []
    for plan, fields, production, table in zip(
        planet_plans, table_records(buildings), table_records(planets), configurations.values()
    ):
        own_warehouses = warehouses[warehouses["slot_index"] == plan["slot_index"]]
        records.append(
            {
                **{column: plan[column] for column in identity},
                "is_central_planet": plan["is_central_planet"],
                "pays_for_research": plan["pays_for_research"],
                "temperature_c": plan["temperature_c"],
                **{column: fields[column] for column in ("fields_max", "fields_used", "fields_free")},
                "production": {column: value for column, value in production.items() if column not in identity},
                "warehouses": table_records(own_warehouses.drop(columns=["slot_index", "coords"])),
                "buildings": table_records(table),
            }
        )
    return json_ready(records)


# The whole result as one JSON document. Every run of the notebook replaces the file.
CONFIG_RESULT: dict[str, Any] = json_ready(
    {
        "description": (
            "Result of notebooks/config_optimizer.ipynb: the building levels of every planet that build the most "
            "target ships per month, as its search found them. The last cell of the notebook writes this file, "
            "and every run replaces it."
        ),
        "schema_version": 1,
        "generated_at": datetime.datetime.now().astimezone().isoformat(timespec="seconds"),
        "meta_parameters": {
            "TARGET_SHIP_KEY": TARGET_SHIP_KEY,
            "PRODUCTION_MONTH_DAYS": PRODUCTION_MONTH_DAYS,
            "RESEARCH_PERCENT_OF_UNIVERSE_MAX": RESEARCH_PERCENT_OF_UNIVERSE_MAX,
            "MIN_FLEET_PER_PLANET": MIN_FLEET_PER_PLANET,
            "MISC_BUILDING_LEVELS": MISC_BUILDING_LEVELS,
            "MISC_BUILDING_LEVELS_BY_SLOT": MISC_BUILDING_LEVELS_BY_SLOT,
            "WAREHOUSE_HOURS_OF_PRODUCTION": WAREHOUSE_HOURS_OF_PRODUCTION,
            "CENTRAL_PLANET_SLOT": CENTRAL_PLANET_SLOT,
            "CENTRAL_PLANET_MIN_WAREHOUSE_LEVEL": CENTRAL_PLANET_MIN_WAREHOUSE_LEVEL,
            "OPTIMIZED_BUILDING_MAX_LEVELS": OPTIMIZED_BUILDING_MAX_LEVELS,
            "CATALOGUE_GRID_SIZE": CATALOGUE_GRID_SIZE,
            "CATALOGUE_PRIORITY_STEPS": CATALOGUE_PRIORITY_STEPS,
            "HILL_CLIMB_STEP_SIZES": HILL_CLIMB_STEP_SIZES,
            "GA_POPULATION_SIZE": GA_POPULATION_SIZE,
            "GA_GENERATIONS": GA_GENERATIONS,
            "GA_TOURNAMENT_SIZE": GA_TOURNAMENT_SIZE,
            "GA_CROSSOVER_RATE": GA_CROSSOVER_RATE,
            "GA_MUTATION_RATE": GA_MUTATION_RATE,
            "GA_ELITE_COUNT": GA_ELITE_COUNT,
            "GA_RANDOM_SEED": GA_RANDOM_SEED,
        },
        "inputs": {
            "universe_speedup_factor": GAME_PARAM_CONTEXT["speedup_factor"],
            "owned_planets": len(live_planet_states),
            "roster_planets": len(PLANET_PLANS),
            "own_research_levels": colony_researches,
            "universe_research_max_levels": UNIVERSE_RESEARCH_MAX_LEVELS,
        },
        "planned_research_levels": PLANNED_RESEARCH_LEVELS,
        "summary": BEST_CONFIG_SUMMARY,
        "search": GA_SUMMARY,
        "limits": table_records(best_config_limits_df),
        "planets": planet_result_records(
            PLANET_PLANS,
            best_config_buildings_df,
            best_config_planets_df,
            best_config_warehouses_df,
            PLANET_CONFIGURATIONS,
        ),
    }
)
CONFIG_RESULT_PATH: Path = METADATA_DIR / CONFIG_RESULT_FILE_NAME
CONFIG_RESULT_PATH.write_text(
    json.dumps(CONFIG_RESULT, indent=2, ensure_ascii=False, allow_nan=False) + "\n", encoding="utf-8"
)

print_section_data(
    f"Section 5 data: result saved as {CONFIG_RESULT_FILE_NAME}",
    [
        ("CONFIG_RESULT_PATH", CONFIG_RESULT_PATH),
        (
            "CONFIG_RESULT (the saved document in short; Sections 4.4 and 4.5 print its tables in full)",
            {
                key: value if isinstance(value, (str, int, float)) else describe_data_object(value)
                for key, value in CONFIG_RESULT.items()
            },
        ),
    ],
)